# Strategic Customer Churn Prediction & Cost-Optimal Retention

## Executive Summary & Business Objective
Customer attrition directly erodes recurring monthly revenue and increases Customer Acquisition Costs (CAC). Traditional machine learning churn projects often optimize solely for statistical metrics (like raw Accuracy or ROC-AUC). In commercial operations, these metrics fail because they treat all errors as equal.

This notebook builds an **economically grounded decision pipeline** designed to:
1. **Identify High-Risk Accounts:** Predict customer churn probabilities with zero data leakage.
2. **Quantify Commercial Trade-offs:** Balance the **\$700 loss of Customer Lifetime Value (CLV)** from a missed churner against the **\$150 expense of a proactive retention concession**.
3. **Optimize Decision Cutoffs ($t^*$):** Replace the arbitrary default threshold of 0.50 with a mathematically derived intervention policy that maximizes net enterprise profit.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

DATA_PATH = os.getenv(
    "TELCO_DATA_PATH", "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)
df = pd.read_csv(DATA_PATH)

---
## 1. Data Ingestion & Schema Sanitization

### Business Context
Raw customer relationship management (CRM) and billing records often arrive with trailing spaces, mixed casing, and non-standard missing values. Establishing clean data contracts early prevents silent pipeline failures downstream.

### Key Operations
* **Record Linkage:** Normalizes `customerID` casing to uppercase and deduplicates rows to guarantee one profile per customer.
* **Financial Type Coercion:** Converts `TotalCharges` from raw text into floating-point numbers; blanks and whitespace are converted to missing values (`NaN`) for transparent auditability.
* **Boolean Normalization:** Translates binary indicator fields (`Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `Churn`, `SeniorCitizen`) into standard boolean flags (`True`/`False`).

In [ ]:
print(df.shape)
df.head()

In [ ]:
obj_cols = df.select_dtypes("object").columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())
df["customerID"] = df["customerID"].str.upper()
df = df.drop_duplicates(subset="customerID", keep="first")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

bool_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
df[bool_cols] = df[bool_cols].replace({"Yes": True, "No": False}).astype(bool)
df["SeniorCitizen"] = df["SeniorCitizen"].astype(bool)  # 0/1 -> False/True

df[["customerID"] + bool_cols + ["TotalCharges"]].head()

### 1.1 Category Whitelisting & Quality Thresholds
* **Operational Risk:** Free-text fields and upstream form inputs frequently introduce typos (e.g., misspelled contract types).
* **Validation Rule:** Defines strict whitelists of valid service tiers. Values outside approved categories are flagged as invalid, and features with more than 50% missing data are pruned immediately to maintain high data density.

In [ ]:
allowed = {
    "gender": ["Female", "Male"],
    "MultipleLines": ["No phone service", "No", "Yes"],
    "InternetService": ["DSL", "Fiber optic", "No"],
    "Contract": ["Month-to-month", "One year", "Two year"],
    "PaymentMethod": ["Electronic check", "Mailed check",
                      "Bank transfer (automatic)", "Credit card (automatic)"],
    **{c: ["No", "Yes", "No internet service"] for c in
       ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies"]},
}

# Any value not in the list becomes empty
for c, vals in allowed.items():
    df.loc[~df[c].isin(vals), c] = np.nan

# Drop columns that are more than 50% empty
df = df.loc[:, df.isna().mean() <= 0.5]

---
## 2. Data Partitioning

### Business Context
The most common pitfall in customer analytics is **data leakage**, allowing information from future holdout records to influence training preprocessing. Evaluating models on leaked data creates misleading backtest performance that fails in live production.

### Partitioning Strategy
* **70 / 30 Stratified Split:** Preserves the natural attrition distribution across both training (learning ground) and test (blind validation) partitions.
* **Strict Boundary:** All summary statistics, bin thresholds, and encoding dictionaries are calculated **exclusively on the training partition** and applied downstream to holdout customers.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["Churn"],
    shuffle=True,
)

df_train = df_train.copy()
df_test  = df_test.copy()

# Sanity check
print(f"Train: {df_train.shape}  Test: {df_test.shape}")
print(f"Train churn rate: {df_train['Churn'].mean():.4f}")
print(f"Test  churn rate: {df_test['Churn'].mean():.4f}")
print(f"Full  churn rate: {df['Churn'].mean():.4f}")

---
## 3. Imputation & Outlier Governance

### Business Context
Improper imputation can distort customer risk profiles. For example, customers with zero months of tenure naturally have missing or zero total charges. We replace missing records using median statistics and bound extreme numerical anomalies using the Interquartile Range (IQR).

### Methodological Safeguards
* **Global vs. Modeling Separation:** Global statistics are calculated strictly for exploratory portfolio reporting.
* **Modeling Imputation:** The modeling pipeline uses training-derived medians and IQR caps to ensure the holdout set remains completely unexposed.

In [ ]:
# Imputation for EDA (df ONLY)
global_medians = {
    "MonthlyCharges": float(df["MonthlyCharges"].median()),
    "tenure": int(df["tenure"].median()),
    "TotalCharges": 0.0,
}

global_iqr_bounds = {}
for c in ["tenure", "MonthlyCharges", "TotalCharges"]:
  q1, q3 = df[c].quantile([0.25, 0.75])
  iqr = q3 - q1
  global_iqr_bounds[c] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# Impute & Clip df using the true global statistics
df["TotalCharges"] = df["TotalCharges"].fillna(global_medians["TotalCharges"])
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(
    global_medians["MonthlyCharges"]
)
df["tenure"] = df["tenure"].fillna(global_medians["tenure"]).astype(int)
for c in ["tenure", "MonthlyCharges", "TotalCharges"]:
  low, high = global_iqr_bounds[c]
  df[c] = df[c].clip(low, high)


# Train-Derived Parameters
train_medians = {
    "MonthlyCharges": float(df_train["MonthlyCharges"].median()),
    "tenure": int(df_train["tenure"].median()),
    "TotalCharges": 0.0,
}

train_iqr_bounds = {}
for c in ["tenure", "MonthlyCharges", "TotalCharges"]:
  q1, q3 = df_train[c].quantile([0.25, 0.75])
  iqr = q3 - q1
  train_iqr_bounds[c] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# Impute & Clip df_train and df_test
for d in [df_train, df_test]:
  d["TotalCharges"] = d["TotalCharges"].fillna(train_medians["TotalCharges"])
  d["MonthlyCharges"] = d["MonthlyCharges"].fillna(
      train_medians["MonthlyCharges"]
  )
  d["tenure"] = d["tenure"].fillna(train_medians["tenure"]).astype(int)
  for c in ["tenure", "MonthlyCharges", "TotalCharges"]:
    low, high = train_iqr_bounds[c]
    d[c] = d[c].clip(low, high)

print(
    f"Cleaned Partitions: {len(df_train):,} Train | {len(df_test):,} Test |"
    f" {len(df):,} Overall"
)
print(f"Overall Medians (For EDA)   : {global_medians}")
print(f"Train Modeling Medians      : {train_medians}")

In [ ]:
print(df.shape)
df.head()

In [ ]:
print(df_train.shape)
df_train.head()

In [ ]:
print(df_test.shape)
df_test.head()

---
## 4. Cohort Overview & Churn Distribution Dynamics

### Business Context
Before building predictive models, leadership needs a macro-level baseline: What is our organic attrition rate, and what do churn curves look like across key account dimensions?

### Strategic Insights
* **Portfolio Attrition Baseline (~26.5% Churn):** The account base exhibits a ~2.77 : 1 non-churn to churn ratio. Because of this class imbalance, simple accuracy is a misleading metric; a baseline that predicts "No Churn" for every customer would appear 73.5% accurate while losing 100% of departing accounts.
* **Tenure Vulnerability:** Accounts in their first 1–6 months experience the highest churn volatility. Retention resources yield the highest marginal ROI when focused early in the lifecycle.
* **Price Sensitivity:** Churn risk increases with higher monthly invoices, highlighting pricing friction points in premium packages.

In [ ]:
n_total = len(df)
n_churn = int(df["Churn"].sum())
n_retain = n_total - n_churn
pct_churn = n_churn / n_total * 100
pct_retain = n_retain / n_total * 100

print(f"""Total Data: {n_total},
Churned Amount: {n_churn} ({pct_churn:.2f}%),
Retained Amount: {n_retain} ({pct_retain:.2f}%)
""")

imbalance_ratio = n_retain / n_churn if n_churn > 0 else np.nan
ratio_imbalance = (
    max(pct_retain, pct_churn) / min(pct_retain, pct_churn)
    if min(pct_retain, pct_churn) > 0 else np.inf
)

print(f"Imbalance: {ratio_imbalance:.2f} : 1")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

NUMERIC_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]
PALETTE = {
    0: "#2b5c8f",  # Deep Blue
    1: "#d95f02",  # Warm Orange
    2: "#10b981",  # Emerald Green
    3: "#f59e0b",  # Amber
    4: "#6366f1",  # Indigo
    5: "#8b5cf6",  # Purple
    6: "#06b6d4",  # Cyan
    7: "#ec4899",  # Rose Pink
    8: "#94a3b8",  # Slate Gray (Baseline)
    9: "#14b8a6",  # Teal
}
sns.set_theme(style="whitegrid", font_scale=1.0)

mosaic = [
    ["pie"] + [f"hist_{col}" for col in NUMERIC_COLS],
    ["pie"] + [f"curve_{col}" for col in NUMERIC_COLS],
]

fig, axes = plt.subplot_mosaic(
    mosaic,
    figsize=(6 * len(NUMERIC_COLS) + 5, 10),
    gridspec_kw={"width_ratios": [1.3] + [1.0] * len(NUMERIC_COLS)},
)

# Imbalance Chart ---
ax_pie = axes["pie"]
ax_pie.pie(
    [n_retain, n_churn],
    labels=[
        f"Retention\n({pct_retain:.1f}%)\nn = {n_retain:,}",
        f"Churn\n({pct_churn:.1f}%)\nn = {n_churn:,}",
    ],
    startangle=90,
    colors=[PALETTE[0], PALETTE[1]],
    wedgeprops={"edgecolor": "white", "linewidth": 2.5, "width": 0.42},
    textprops={"fontsize": 11},
)
ax_pie.text(
    0, 0,
    f"Total N = {n_total:,}\nRatio: {imbalance_ratio:.2f} : 1",
    ha="center", va="center", fontsize=11, weight="bold",
)
ax_pie.set_title("Retention vs. Churn", weight="bold", fontsize=14)

# Density & Empirical Churn Rate ---
for col in NUMERIC_COLS:
    ax_hist = axes[f"hist_{col}"]
    ax_curve = axes[f"curve_{col}"]

    sns.histplot(
        data=df, x=col, hue="Churn", kde=True, ax=ax_hist,
        palette={False: PALETTE[0], True: PALETTE[1]}, stat="density",
        common_norm=False, bins=30, alpha=0.4,
    )
    ax_hist.set_title(col, weight="bold", fontsize=12)
    ax_hist.set_xlabel("")
    ax_hist.legend(["Retention", "Churn"], loc="upper right")

    # Bottom Row: Empirical churn rate curve
    bins = pd.qcut(df[col], q=20, duplicates="drop")
    churn_rates = df.groupby(bins, observed=True)["Churn"].mean() * 100
    bin_centers = df.groupby(bins, observed=True)[col].mean()

    ax_curve.plot(bin_centers, churn_rates, marker="o", linewidth=2.5,
                  color=PALETTE[1], label="Local Churn Rate (%)")
    ax_curve.axhline(y=df["Churn"].mean() * 100, color="gray", linestyle="--",
                     linewidth=1.5, label=f"Avg Churn ({df['Churn'].mean()*100:.1f}%)")
    ax_curve.set_title(f"{col}: Churn Rate Curve", weight="bold", fontsize=12)
    ax_curve.set_xlabel(col, weight="bold")
    ax_curve.set_ylabel("Churn Rate (%)", weight="bold")
    ax_curve.legend(loc="best")
    ax_curve.set_ylim(0, max(churn_rates.max() * 1.15, 45))

fig.suptitle("Overview & Numeric Feature Distributions", weight="bold", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df_train.head()

---
## 5. Association & Collinearity Auditing

### Business Context
Redundant features inflate operational complexity, introduce statistical collinearity, and make model decisions harder to explain to business stakeholders. We evaluate relationships across two dimensions:
1. **Pearson Correlation (Numeric):** Measures linear co-movement between financial metrics (`MonthlyCharges`, `TotalCharges`, `tenure`).
2. **Cramér's V (Categorical):** Quantifies non-linear associations among service offerings, contract types, and churn outcomes.

In [ ]:
MAX_COLS  = 20
MAX_ANNOT = 15
TARGET    = "Churn"

if TARGET not in df_train.columns:
    raise KeyError(f"Target '{TARGET}' not found in df_train.")

df_fe = df_train.copy()

num_cols = [
    c for c in df_fe.select_dtypes(include=np.number).columns
    if c != TARGET and not pd.api.types.is_bool_dtype(df_fe[c])
]
cat_cols = [
    c for c in df_fe.select_dtypes(include=["object", "category", "bool"]).columns
    if c != TARGET
]

num_features = num_cols[:MAX_COLS]
cat_features = ([TARGET] + cat_cols)[:MAX_COLS]
corr = (
    df_fe[num_features].corr()
    if len(num_features) >= 2
    else pd.DataFrame()
)

cramer = pd.DataFrame()
if len(cat_features) >= 2:
    codes = {c: pd.factorize(df_fe[c], sort=True)[0] for c in cat_features}
    n = len(cat_features)
    mat = np.eye(n)
    for i, c1 in enumerate(cat_features):
        for j in range(i + 1, n):
            a, b = codes[c1], codes[cat_features[j]]
            m = (a >= 0) & (b >= 0)             # -1 = NaN from pd.factorize
            a, b = a[m], b[m]
            if a.size < 2:
                v = 0.0
            else:
                ka, kb = a.max() + 1, b.max() + 1
                if ka < 2 or kb < 2:
                    v = 0.0
                else:
                    ct = np.bincount(a * kb + b, minlength=ka * kb).reshape(ka, kb)
                    nn = ct.sum()
                    exp = ct.sum(1, keepdims=True) @ ct.sum(0, keepdims=True) / nn
                    with np.errstate(divide="ignore", invalid="ignore"):
                        chi2 = np.nansum((ct - exp) ** 2 / exp)
                    r, k = ct.shape
                    phi2 = max(0.0, chi2 / nn - ((k - 1) * (r - 1)) / (nn - 1))
                    denom = min(
                        k - ((k - 1) ** 2) / (nn - 1) - 1,
                        r - ((r - 1) ** 2) / (nn - 1) - 1,
                    )
                    v = float(np.sqrt(phi2 / denom)) if denom > 0 else 0.0
            mat[i, j] = mat[j, i] = v
    cramer = pd.DataFrame(mat, index=cat_features, columns=cat_features)

# plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, mat, title, cmap, fmt, vmin, vmax in [
    (axes[0], corr,
     f"Pearson correlation — numeric (n={len(num_features)})",
     "vlag",  ".3f", -1, 1),
    (axes[1], cramer,
     f"Cramér's V — categorical (n={len(cat_features)}, target='{TARGET}')",
     "Blues", ".2f",  0, 1),
]:
    if mat.empty:
        ax.text(.5, .5, "Not enough features", ha="center", va="center",
                fontsize=12, weight="bold")
        ax.set_axis_off()
        continue
    annot = mat.shape[0] <= MAX_ANNOT
    sns.heatmap(mat, annot=annot, fmt=fmt, cmap=cmap, vmin=vmin, vmax=vmax,
                ax=ax, linewidths=1.0, cbar_kws={"shrink": .85},
                annot_kws={"size": 8} if annot else None)
    ax.set_title(title, weight="bold", fontsize=12)
    ax.tick_params(axis="x", rotation=30, labelsize=9)
    plt.setp(ax.get_xticklabels(), ha="right", weight="bold")
    plt.setp(ax.get_yticklabels(), weight="bold")

plt.tight_layout()
plt.show()

In [ ]:
from itertools import combinations

CORR_DROP_THRESH   = 0.85
CRAMER_DROP_THRESH = 0.60
CRAMER_WEAK_THRESH = 0.025

y = df_fe[TARGET].astype(int)

# Numerical collinearity
drop_numeric = set()
for c1, c2 in combinations(num_features, 2):
    if abs(corr.loc[c1, c2]) >= CORR_DROP_THRESH:
        r1 = abs(y.corr(df_fe[c1]))
        r2 = abs(y.corr(df_fe[c2]))
        drop_numeric.add(c2 if r2 <= r1 else c1)

# Categorical redundancy & weak signals
cat_feats_only = [c for c in cat_features if c != TARGET]
drop_cat, review = set(), []
for c1, c2 in combinations(cat_feats_only, 2):
    if cramer.loc[c1, c2] >= CRAMER_DROP_THRESH:
        v1, v2 = cramer.loc[c1, TARGET], cramer.loc[c2, TARGET]
        if min(v1, v2) >= CRAMER_WEAK_THRESH and abs(v1 - v2) < 0.05:
            review.append(f"{c1} | {c2}")
        else:
            drop_cat.add(c2 if v2 <= v1 else c1)

weak_cat = [c for c in cat_feats_only if cramer.loc[c, TARGET] < CRAMER_WEAK_THRESH]

# SUMMARY
print(f"  Drop (numerical collinearity) : {sorted(drop_numeric) or 'none'}")
print(f"  Drop (categorical redundancy) : {sorted(drop_cat) or 'none'}")
print(f"  Drop (weak signal)            : {sorted(weak_cat) or 'none'}")
print(f"  Review manually               : {review or 'none'}")

### 5.1 Deep Feature Audit: The Billing Identity & Add-on Nesting
* **The Accounting Identity:** A subscriber's cumulative bill (`TotalCharges`) is largely the mathematical product of account age and subscription price ($\text{tenure} \times \text{MonthlyCharges}$). An $R^2 \approx 1.0$ indicates that cumulative billing provides no net-new behavioral insight beyond tenure and monthly price.
* **Structural Dependency:** Subscribers without internet service (`InternetService == 'No'`) automatically generate redundant "No internet service" flags across six supplementary features (`OnlineSecurity`, `StreamingTV`, etc.). Identifying these patterns lets us simplify the feature set without losing predictive power.

In [ ]:
prod     = df_fe["tenure"] * df_fe["MonthlyCharges"]
r        = df_fe["TotalCharges"].corr(prod)
diff     = (df_fe["TotalCharges"] - prod).abs()
rel_diff = (diff / (df_fe["TotalCharges"] + 1e-5)).mean() * 100

addon_cols = [c for c in df_fe.columns if (df_fe[c] == "No internet service").any()]

print("=" * 70 + "\nCOLLINEARITY AUDIT: TotalCharges vs. (tenure * MonthlyCharges)\n" + "=" * 70)
print(f"Pearson Correlation (r)           : {r:.6f}")
print(f"Coefficient of Determination (R²) : {r**2:.6f}")
print(f"Mean Absolute Difference ($)      : ${diff.mean():.2f} (Median: ${diff.median():.2f})")
print(f"Mean Relative Difference (%)      : {rel_diff:.2f}%\n" + "=" * 70)

print("CATEGORY OVERLAP AUDIT: Add-on Values when InternetService == 'No':")
for col in addon_cols:
    counts = df_fe.loc[df_fe["InternetService"] == "No", col].value_counts().to_dict()
    print(f"  • {col:<20} -> {counts}")
print("=" * 70)

---
## 6. Systematic Feature Pruning & Ablation Architecture

### Business Context
Leaner models require less maintenance, generalize better against distribution shifts, and provide clearer explanations for commercial teams. We test five candidate data configurations:

| Configuration | Structural Treatment | Business Objective |
| :--- | :--- | :--- |
| **1. Raw Baseline** | Full unedited feature space | Benchmark reference |
| **2. Filter Low-Signal** | Exclude zero-correlation features | Reduce background noise |
| **3. Prune Add-on Redundancy** | Remove duplicate add-on flags | Eliminate redundant service flags |
| **4. Feature Compression** | Aggregate add-ons into a total service count | Measure customer stickiness via product adoption |
| **5. Binary Collapsing** | Prune `TotalCharges` + collapse binary service flags | Maximize model simplicity and operational stability |

In [ ]:
# Base modeling frame — features only (target excluded)
c1 = df_fe.drop(columns=[c for c in ["customerID", TARGET] if c in df_fe.columns])

low_signal = [c for c in weak_cat if c in c1.columns]

addon_cols = [
    c for c in c1.columns
    if df_fe[c].astype(str).eq("No internet service").any()
]
redundant_addons = [c for c in addon_cols if c in drop_cat]
retain_addons    = [c for c in addon_cols if c not in drop_cat]

# 2. Incremental pruning
c2 = c1.drop(columns=low_signal, errors="ignore")
c3 = c2.drop(columns=redundant_addons, errors="ignore")

# Service count must come from the ORIGINAL add-on columns (before collapsing)
c4 = c3.assign(num_internet_services=(df_fe[addon_cols] == "Yes").sum(axis=1))

# Full pruning: drop collinear TotalCharges; collapse retained add-ons to 0/1
c5 = (
    c4.drop(columns=["TotalCharges"], errors="ignore")
      .assign(**{c: (df_fe[c] == "Yes").astype(int) for c in retain_addons})
)

candidate_pruning_configs = [
    ("1. Raw Baseline",                     c1),
    ("2. Filter Low-Signal",                c2),
    ("3. Prune Add-on Redundancy",          c3),
    ("4. Prune Add-ons + Service Count",    c4),
    ("5. Full Pruning + Binary Collapsing", c5),
]

print("Candidate Pruning Configurations Constructed:")
for name, cfg in candidate_pruning_configs:
    print(f"  {name:<38} : {cfg.shape[1]} raw features")

### 6.1 Ablation Cross-Validation (5-Fold Stratified CV)
* Evaluates both a parametric linear classifier (**Logistic Regression**) and an ensemble decision tree (**Random Forest**) across all five configurations.
* Tracks out-of-fold generalization performance (ROC-AUC) to confirm whether reducing features preserves predictive power.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 5-Fold Stratified Cross-Validation Ablation Experiment
y  = df_train[TARGET].astype(int).reset_index(drop=True)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
records = []

for name, sub in candidate_pruning_configs:
    sub = sub.reset_index(drop=True)                       # align with y
    X = pd.get_dummies(
        sub,
        columns=sub.select_dtypes(exclude="number").columns,
        drop_first=True,
    )

    # Scale every numeric column that isn't a 0/1 dummy:
    # tenure, MonthlyCharges, TotalCharges, interaction products,
    # num_internet_services, and any future count/continuous feature.
    scale_cols = [
        c for c in X.columns
        if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 2
    ]

    lr_sc, rf_sc = [], []

    for tr, val in cv.split(X, y):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[val].copy()

        if scale_cols:
            scl = StandardScaler()
            X_tr[scale_cols] = scl.fit_transform(X_tr[scale_cols])
            X_va[scale_cols] = scl.transform(X_va[scale_cols])

        lr = LogisticRegression(
            max_iter=5000, random_state=42,
        ).fit(X_tr, y.iloc[tr])
        rf = RandomForestClassifier(
            n_estimators=100, max_depth=8,
            random_state=42, n_jobs=-1,
        ).fit(X_tr, y.iloc[tr])

        lr_sc.append(roc_auc_score(y.iloc[val], lr.predict_proba(X_va)[:, 1]))
        rf_sc.append(roc_auc_score(y.iloc[val], rf.predict_proba(X_va)[:, 1]))

    (lr_m, lr_se), (rf_m, rf_se) = [
        (np.mean(s), np.std(s, ddof=1) / np.sqrt(len(s))) for s in (lr_sc, rf_sc)
    ]
    records.append({
        "Configuration": name,
        "Raw Features": sub.shape[1],
        "Encoded Dims": X.shape[1],
        "LR 5-Fold AUC": f"{lr_m:.4f} ± {lr_se:.4f}",
        "RF 5-Fold AUC": f"{rf_m:.4f} ± {rf_se:.4f}",
        "lr_mean": lr_m, "lr_se": lr_se,
        "rf_mean": rf_m, "rf_se": rf_se,
    })

ablation_df = pd.DataFrame(records)
display(ablation_df[["Configuration", "Raw Features", "Encoded Dims",
                     "LR 5-Fold AUC", "RF 5-Fold AUC"]])

# Visualization Pruning Configurations
fig, ax = plt.subplots(figsize=(13, 5.5), dpi=150)
labels = [
    "1. Raw Baseline\n(All Features)",
    "2. Filter Low-Signal",
    "3. Prune Redundancy\n(Add-on Subset)",
    "4. Prune Add-ons\n+ Service Count",
    "5. Full Pruning +\nBinary Collapsing",
]
x, w = np.arange(len(labels)), 0.32

rects_lr = ax.bar(
    x - w / 2, ablation_df["lr_mean"], w,
    yerr=ablation_df["lr_se"], capsize=4,
    label="Logistic Regression", color=PALETTE[0], alpha=0.88, ec="black", lw=0.8,
)
rects_rf = ax.bar(
    x + w / 2, ablation_df["rf_mean"], w,
    yerr=ablation_df["rf_se"], capsize=4,
    label="Random Forest", color=PALETTE[1], alpha=0.88, ec="black", lw=0.8,
)

base_m, base_se = ablation_df.loc[0, ["lr_mean", "lr_se"]]
ax.axhline(base_m, color=PALETTE[0], ls="--", lw=1.2, alpha=0.7,
           label=f"LR Baseline ({base_m:.4f})")
ax.axhspan(base_m - base_se, base_m + base_se, color=PALETTE[0], alpha=0.08,
           label=f"LR Baseline ±1 SE ({base_se:.4f})")

ax.set(ylim=(0.830, 0.860), xticks=x, xticklabels=labels)
ax.set_ylabel("5-Fold Stratified CV ROC-AUC (Mean ± 1 SE)", weight="bold", fontsize=11)
ax.set_title(
    "5-Fold Stratified Cross-Validation Across Feature Pruning Configurations",
    weight="bold", fontsize=12, pad=12,
)
plt.setp(ax.get_xticklabels(), weight="bold", fontsize=9.5)
ax.grid(axis="y", ls=":", alpha=0.6)
ax.legend(loc="lower right", framealpha=0.95, fontsize=8.5)

for rects, m_col, s_col in [(rects_lr, "lr_mean", "lr_se"),
                            (rects_rf, "rf_mean", "rf_se")]:
    for r, m, s in zip(rects, ablation_df[m_col], ablation_df[s_col]):
        ax.annotate(
            f"{m:.4f}\n±{s:.4f}",
            (r.get_x() + r.get_width() / 2, r.get_height() + s),
            xytext=(0, 4), textcoords="offset points",
            ha="center", va="bottom", fontsize=8, weight="bold",
        )

plt.tight_layout()
plt.show()

# Select the optimal configuration by composite mean AUC ---
if "avg_mean" not in ablation_df.columns:
    ablation_df["avg_mean"] = (ablation_df["lr_mean"] + ablation_df["rf_mean"]) / 2

opt_pos          = ablation_df.sort_values(["avg_mean", "Encoded Dims"],
                                           ascending=[False, True]).index[0]
opt_name, opt_df = candidate_pruning_configs[opt_pos]
df_base          = opt_df.reset_index(drop=True).assign(**{TARGET: y.values})

row = ablation_df.loc[opt_pos]
print("=" * 70)
print(f"OPTIMAL CONFIGURATION SELECTED: {opt_name}")
print("=" * 70)
print(f"Dataset Dimensions     : {df_base.shape[0]:,} rows x {df_base.shape[1]} columns")
print(f"Modeling Features Count: {df_base.shape[1] - 1} (+ '{TARGET}')")
print(f"Validation LR AUC      : {row['lr_mean']:.4f}")
print(f"Validation RF AUC      : {row['rf_mean']:.4f}")
print(f"Composite Mean AUC     : {row['avg_mean']:.4f}")
print("=" * 70)
print("Retained Features & Statistical Modalities:")
for i, col in enumerate(df_base.columns):
    if col == TARGET:
        continue
    s   = df_base[col]
    mod = (
        "Continuous" if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
        else "Boolean" if pd.api.types.is_bool_dtype(s)
        else "Categorical"
    )
    print(f"  {i+1:<3} | {col:<24} | Type: {str(s.dtype):<10} | Modality: {mod}")
print("=" * 70)

---
## 7. Permutation Importance & LOFO

### Business Context
Single-model importance scores can be misleading when correlated features share predictive credit. We combine two validation methods to confirm feature importance:
* **Permutation Importance:** Measures the drop in AUC when a feature is randomly shuffled.
* **Leave-One-Feature-Out (LOFO):** Retrains models with each feature withheld, directly quantifying out-of-sample reliance.

### Pruning Rule
A feature is recommended for removal only if **both models show zero deterioration in out-of-sample ROC-AUC** ($\Delta\text{AUC} \le 0$) when that feature is omitted.

In [ ]:
from sklearn.inspection import permutation_importance

feature_cols = [c for c in df_base.columns if c != TARGET]
X_base = pd.get_dummies(df_base[feature_cols], drop_first=True)
y_base = df_base[TARGET].astype(int).reset_index(drop=True)   # positional index

cont = [
    c for c in X_base.columns
    if pd.api.types.is_numeric_dtype(X_base[c]) and X_base[c].nunique() > 2
]

# Permutation importance
scl   = StandardScaler()
X_scl = X_base.copy()
if cont:
    X_scl[cont] = scl.fit_transform(X_scl[cont])

lr_full = LogisticRegression(max_iter=5000, random_state=42).fit(X_scl, y_base)
rf_full = RandomForestClassifier(
    n_estimators=100, max_depth=8, random_state=42, n_jobs=-1,
).fit(X_scl, y_base)

lr_perm = permutation_importance(
    lr_full, X_scl, y_base, scoring="roc_auc", n_repeats=5, random_state=42,
)
rf_perm = permutation_importance(
    rf_full, X_scl, y_base, scoring="roc_auc", n_repeats=5, random_state=42,
)

# Aggregate one-hot dummies back to raw features
perm_records = []
for raw_feat in feature_cols:
    idx = [
        i for i, col in enumerate(X_base.columns)
        if col == raw_feat or col.startswith(f"{raw_feat}_")
    ]
    perm_records.append({
        "Feature": raw_feat,
        "LR Importance (ΔAUC)": lr_perm.importances_mean[idx].sum(),
        "RF Importance (ΔAUC)": rf_perm.importances_mean[idx].sum(),
    })

perm_summary_df = (
    pd.DataFrame(perm_records)
      .sort_values("RF Importance (ΔAUC)", ascending=False)
      .reset_index(drop=True)
)
display(perm_summary_df.round(4))

# Leave-One-Feature-Out (LOFO) ablation
base_lr_auc = ablation_df.loc[opt_pos, "lr_mean"]
base_rf_auc = ablation_df.loc[opt_pos, "rf_mean"]
lofo_records = []

for feat in perm_summary_df["Feature"]:
    sub_lofo = df_base.drop(columns=[TARGET, feat]).reset_index(drop=True)
    X_lofo   = pd.get_dummies(sub_lofo, drop_first=True)
    c_lofo   = [
        c for c in X_lofo.columns
        if pd.api.types.is_numeric_dtype(X_lofo[c]) and X_lofo[c].nunique() > 2
    ]

    lr_sc, rf_sc = [], []
    for tr, va in cv.split(X_lofo, y_base):
        X_tr, X_va = X_lofo.iloc[tr].copy(), X_lofo.iloc[va].copy()

        if c_lofo:
            s = StandardScaler()
            X_tr[c_lofo] = s.fit_transform(X_tr[c_lofo])
            X_va[c_lofo] = s.transform(X_va[c_lofo])

        lr = LogisticRegression(max_iter=5000, random_state=42).fit(X_tr, y_base.iloc[tr])
        rf = RandomForestClassifier(
            n_estimators=100, max_depth=8, random_state=42, n_jobs=-1,
        ).fit(X_tr, y_base.iloc[tr])

        lr_sc.append(roc_auc_score(y_base.iloc[va], lr.predict_proba(X_va)[:, 1]))
        rf_sc.append(roc_auc_score(y_base.iloc[va], rf.predict_proba(X_va)[:, 1]))

    lofo_records.append({
        "Withheld Feature": feat,
        "LR ΔAUC": base_lr_auc - np.mean(lr_sc),
        "RF ΔAUC": base_rf_auc - np.mean(rf_sc),
    })

lofo_results_df = (
    pd.DataFrame(lofo_records)
      .sort_values("RF ΔAUC", ascending=False)
      .reset_index(drop=True)
)
display(lofo_results_df.round(4))

In [ ]:
# Merge into one frame so both panels share the same row order
merged = (
    perm_summary_df[["Feature", "LR Importance (ΔAUC)", "RF Importance (ΔAUC)"]]
    .merge(
        lofo_results_df[["Withheld Feature", "LR ΔAUC", "RF ΔAUC"]]
        .rename(columns={"Withheld Feature": "Feature"}),
        on="Feature", how="outer",
    )
    .fillna(0.0)
)

# Drop rule ---
# A feature is dropped only if BOTH estimators agree its removal
# does not hurt out-of-sample AUC. Averaging hid LR-vs-RF
# disagreement (e.g. tenure: LR +0.007, RF -0.001) and let the
# rule drop features that only LR found redundant due to
# multicollinearity.
TOL = -0.00005
merged["Drop?"] = (
    (merged["LR ΔAUC"] <= TOL) &
    (merged["RF ΔAUC"] <= TOL)
)

# Keep both per-model values for sorting and reporting
merged["Average LOFO ΔAUC"] = (merged["LR ΔAUC"] + merged["RF ΔAUC"]) / 2

# Sort by the average of the two RF signals — LOFO is the honest one.
merged["sort_key"] = (merged["RF Importance (ΔAUC)"] + merged["RF ΔAUC"]) / 2
merged = (
    merged.sort_values("sort_key")
          .drop(columns="sort_key")
          .reset_index(drop=True)
)

y_pos = np.arange(len(merged))
h = 0.38

fig, axes = plt.subplots(1, 2, figsize=(15, 0.55 * len(merged) + 2),
                         dpi=150, sharey=True)

panels = [
    (axes[0], "LR Importance (ΔAUC)", "RF Importance (ΔAUC)",
     "Permutation Importance — in-sample (df_base fit)"),
    (axes[1], "LR ΔAUC", "RF ΔAUC",
     "Leave-One-Feature-Out — 5-fold CV (out-of-sample)"),
]

# Highlight recommended-drop rows with a pale red band across both panels
drop_rows = merged.index[merged["Drop?"]].tolist()

for ax, lr_col, rf_col, title in panels:
    # Highlight drop rows first (behind the bars)
    for i in drop_rows:
        ax.axhspan(i - 0.5, i + 0.5, color="red", alpha=0.08, zorder=0)

    ax.barh(y_pos - h/2, merged[lr_col], h,
            color=PALETTE[0], alpha=0.9, ec="black", lw=0.6,
            label="Logistic Regression")
    ax.barh(y_pos + h/2, merged[rf_col], h,
            color=PALETTE[1], alpha=0.9, ec="black", lw=0.6,
            label="Random Forest")

    ax.axvline(0, color="black", lw=1.0)
    ax.set_yticks(y_pos)

    # Append a marker to labels of drop-recommended features
    labels = [
        f"✗ {f}" if d else f"  {f}"
        for f, d in zip(merged["Feature"], merged["Drop?"])
    ]
    ax.set_yticklabels(labels, weight="bold", fontsize=9.5)
    # Colour the marker itself red
    for tick, d in zip(ax.get_yticklabels(), merged["Drop?"]):
        if d:
            tick.set_color("crimson")

    ax.set_title(title, weight="bold", fontsize=11, pad=10)
    ax.set_xlabel("ΔAUC  (positive = feature matters)", fontsize=10)
    ax.grid(axis="x", ls=":", alpha=0.5)
    ax.legend(loc="lower right", framealpha=0.95, fontsize=9)

# Shade the negative-ΔAUC half so harmful/noise features stand out
for ax in axes:
    xlim = ax.get_xlim()
    ax.axvspan(xlim[0], 0, color="red", alpha=0.05, zorder=0)

plt.suptitle(
    f"Feature Importance: Permutation vs. LOFO  "
    f"(red rows = LR ΔAUC ≤ {TOL:+.3f} AND RF ΔAUC ≤ {TOL:+.3f})",
    weight="bold", fontsize=13, y=1.005,
)
plt.tight_layout()
plt.show()

# summary
rec_drop = (
    merged.loc[merged["Drop?"],
               ["Feature", "LR ΔAUC", "RF ΔAUC", "Average LOFO ΔAUC"]]
          .sort_values("Average LOFO ΔAUC")
          .reset_index(drop=True)
)

print(f"Recommended Drops ({len(rec_drop)}) — "
      f"rule: LR ΔAUC ≤ {TOL:+.3f} AND RF ΔAUC ≤ {TOL:+.3f}")
print("-" * 72)
if rec_drop.empty:
    print("  none — at least one estimator relies on every feature")
else:
    print(f"  {'Feature':<22} | {'LR ΔAUC':>9} | {'RF ΔAUC':>9} | {'Avg':>9}")
    print("  " + "-" * 68)
    for _, r in rec_drop.iterrows():
        print(f"  {r['Feature']:<22} | {r['LR ΔAUC']:>+9.4f} | "
              f"{r['RF ΔAUC']:>+9.4f} | {r['Average LOFO ΔAUC']:>+9.4f}")
print("-" * 72)
print(f"Recommended feature set ({merged.shape[0] - len(rec_drop)} features):")
print("  " + ", ".join(merged.loc[~merged["Drop?"], "Feature"].tolist()))
drop_lofo = merged.loc[merged["Drop?"], "Feature"].tolist()

In [ ]:
base_cols = [c for c in df_base.columns if c != TARGET]
experiments = [("Baseline", base_cols)]
if drop_lofo:
    experiments.append(
        (f"Post-LOFO(-{len(drop_lofo)})",
         [c for c in base_cols if c not in drop_lofo])
    )

# Cross-validated evaluation of each candidate set
y_base = df_base[TARGET].astype(int).reset_index(drop=True)
cv     = StratifiedKFold(5, shuffle=True, random_state=42)
records = []

for name, cols in experiments:
    sub = df_base[cols].reset_index(drop=True)

    X = pd.get_dummies(
        sub,
        columns=sub.select_dtypes(exclude="number").columns,
        drop_first=True,
    )

    # Every non-binary numeric column needs scaling (same rule as ablation cell)
    scale_cols = [
        c for c in X.columns
        if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 2
    ]

    lr_sc, rf_sc = [], []

    for tr, va in cv.split(X, y_base):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[va].copy()
        if scale_cols:
            scl = StandardScaler()
            X_tr[scale_cols] = scl.fit_transform(X_tr[scale_cols])
            X_va[scale_cols] = scl.transform(X_va[scale_cols])

        lr = LogisticRegression(max_iter=5000, random_state=42).fit(X_tr, y_base.iloc[tr])
        rf = RandomForestClassifier(
            n_estimators=100, max_depth=8, random_state=42, n_jobs=-1,
        ).fit(X_tr, y_base.iloc[tr])

        lr_sc.append(roc_auc_score(y_base.iloc[va], lr.predict_proba(X_va)[:, 1]))
        rf_sc.append(roc_auc_score(y_base.iloc[va], rf.predict_proba(X_va)[:, 1]))

    # Mean + SE of the mean (SE = SD / sqrt(n_folds))
    lr_arr, rf_arr = np.array(lr_sc), np.array(rf_sc)
    lr_m, lr_se = lr_arr.mean(), lr_arr.std(ddof=1) / np.sqrt(len(lr_arr))
    rf_m, rf_se = rf_arr.mean(), rf_arr.std(ddof=1) / np.sqrt(len(rf_arr))

    records.append({
        "Configuration":       name,
        "Raw Features":        len(cols),
        "Encoded Dims":        X.shape[1],
        "LR Mean AUC ± SE":    f"{lr_m:.4f} ± {lr_se:.4f}",
        "RF Mean AUC ± SE":    f"{rf_m:.4f} ± {rf_se:.4f}",
        "Overall Average AUC": (lr_m + rf_m) / 2,
        "lr_folds":            lr_sc,
        "rf_folds":            rf_sc,
        "cols":                cols,
    })

res_df = (
    pd.DataFrame(records)
      .sort_values(["Overall Average AUC", "Encoded Dims"],
                   ascending=[False, True])
      .reset_index(drop=True)
)
res_df.insert(0, "Rank", [f"Rank #{i+1}" for i in range(len(res_df))])

# 1-SE rule---
# AUC_TOL = one standard error of the best config's CV mean.
# Configs within this band are statistically indistinguishable from
# the leader; among them, pick the smallest (fewest encoded dims).
best_row  = res_df.iloc[0]
se_leader = max(
    np.std(best_row["lr_folds"], ddof=1) / np.sqrt(len(best_row["lr_folds"])),
    np.std(best_row["rf_folds"], ddof=1) / np.sqrt(len(best_row["rf_folds"])),
)
AUC_TOL = se_leader
best_auc = best_row["Overall Average AUC"]

res_df["AUC Δ vs. best"] = res_df["Overall Average AUC"] - best_auc
tied     = res_df[res_df["Overall Average AUC"] >= best_auc - AUC_TOL]
winner   = tied.sort_values("Encoded Dims").iloc[0]

res_df["Selected?"] = res_df["cols"].apply(lambda c: c == winner["cols"])

# Display
cols_show = ["Rank", "Configuration", "Selected?",
             "Raw Features", "Encoded Dims",
             "LR Mean AUC ± SE", "RF Mean AUC ± SE",
             "Overall Average AUC", "AUC Δ vs. best"]
display(
    res_df[cols_show].assign(**{
        "Overall Average AUC": res_df["Overall Average AUC"].map("{:.4f}".format),
        "AUC Δ vs. best":      res_df["AUC Δ vs. best"].map("{:+.5f}".format),
    })
)

print(f"Leader              : {best_row['Configuration']} "
      f"(AUC = {best_auc:.4f})")
print(f"1-SE band           : [{best_auc - AUC_TOL:.4f}, {best_auc:.4f}]  "
      f"(AUC_TOL = {AUC_TOL:.4f})")
print(f"Configs inside band : {tied['Configuration'].tolist()}")
print(f"Winner (smallest)   : {winner['Configuration']}  "
      f"({winner['Encoded Dims']} encoded dims, AUC = {winner['Overall Average AUC']:.4f})")
if winner["Configuration"] != best_row["Configuration"]:
    print(f"  ↳ {best_row['Configuration']} has the highest point estimate "
          f"but is statistically tied with the winner; preferring the simpler model.")

df_selected = df_base[winner["cols"] + [TARGET]].copy()
print()
print(f"Final Features ({len(df_selected.columns) - 1} Features + Target):")
for i, c in enumerate(df_selected.columns, 1):
    s = df_selected[c]
    print(f"  {i:>2}. {c:<25} | Type: {str(s.dtype):<10} | Nulls: {s.isna().sum()}")

In [ ]:
from sklearn.metrics import roc_curve, auc

# 5-Fold Stratified CV ROC-AUC Validation & Multi-Stage Benchmark
feat_cols = [c for c in df_selected.columns if c != TARGET]
X = pd.get_dummies(
    df_selected[feat_cols],
    columns=df_selected[feat_cols].select_dtypes(exclude="number").columns,
    drop_first=True,
)
y = df_selected[TARGET].astype(int).reset_index(drop=True)

# Same scaling rule as every other CV cell in the notebook
scale_cols = [
    c for c in X.columns
    if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 2
]

mean_fpr = np.linspace(0, 1, 100)
cv       = StratifiedKFold(5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": (LogisticRegression(max_iter=5000, random_state=42),
                            PALETTE[0]),
    "Random Forest":       (RandomForestClassifier(n_estimators=100, max_depth=8,
                                                    random_state=42, n_jobs=-1),
                            PALETTE[1]),
}

# Generalization ROC-AUC Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
perf = {}

for ax, (name, (clf, col)) in zip(axes, models.items()):
    tprs, fold_aucs = [], []

    for i, (tr, va) in enumerate(cv.split(X, y), 1):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[va].copy()

        # Only LR benefits from scaling; RF is invariant to monotone transforms
        if scale_cols and "Logistic" in name:
            scl = StandardScaler()
            X_tr[scale_cols] = scl.fit_transform(X_tr[scale_cols])
            X_va[scale_cols] = scl.transform(X_va[scale_cols])

        clf.fit(X_tr, y.iloc[tr])
        fpr, tpr, _ = roc_curve(y.iloc[va], clf.predict_proba(X_va)[:, 1])
        sc = auc(fpr, tpr)
        fold_aucs.append(sc)
        tprs.append(np.interp(mean_fpr, fpr, tpr))
        tprs[-1][0] = 0.0
        ax.plot(fpr, tpr, lw=1, alpha=0.3, ls="--",
                label=f"Fold {i} ({sc:.4f})")

    m_tpr, m_auc, s_auc = np.mean(tprs, 0), np.mean(fold_aucs), np.std(fold_aucs, ddof=1)
    m_tpr[-1] = 1.0
    perf[name] = (m_auc, s_auc, s_auc / np.sqrt(len(fold_aucs)))

    ax.plot(mean_fpr, m_tpr, color=col, lw=2.2,
            label=f"Mean ROC ({m_auc:.4f} ± {s_auc:.4f})")
    std_tpr = np.std(tprs, 0)
    ax.fill_between(mean_fpr,
                    np.maximum(m_tpr - std_tpr, 0),
                    np.minimum(m_tpr + std_tpr, 1),
                    color="gray", alpha=0.15)
    ax.plot([0, 1], [0, 1], ":", color="gray", lw=1.2)
    ax.set(xlim=[-0.02, 1.02], ylim=[-0.02, 1.02],
           xlabel="False Positive Rate", ylabel="True Positive Rate")
    ax.set_title(f"{name}\n5-Fold CV Empirical ROC Curve", weight="bold", pad=10)
    ax.grid(ls=":", alpha=0.5)
    ax.legend(loc="lower right", fontsize=8.5)

plt.suptitle(
    f"5-Fold Stratified CV ROC Curves: Post-LOFO Refined Set "
    f"({len(df_selected.columns) - 1} Features)",
    weight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

# Multi-Stage Empirical Pruning Leaderboard
b = ablation_df.iloc[0]           
p = ablation_df.iloc[opt_pos]   

s_lr, _, s_lr_se = perf["Logistic Regression"]
s_rf, _, s_rf_se = perf["Random Forest"]
s_avg = (s_lr + s_rf) / 2

b_lr, b_rf = b["lr_mean"], b["rf_mean"]
b_avg      = (b_lr + b_rf) / 2
p_lr, p_rf = p["lr_mean"], p["rf_mean"]
p_avg      = (p_lr + p_rf) / 2

b_raw, b_dims = int(b["Raw Features"]), int(b["Encoded Dims"])
p_raw, p_dims = int(p["Raw Features"]), int(p["Encoded Dims"])
s_raw, s_dims = len(df_selected.columns) - 1, X.shape[1]

comp_df = pd.DataFrame([
    {"Stage": "1. Baseline",
     "Raw": b_raw, "Dims": b_dims,
     "LR AUC": f"{b_lr:.4f} ± {b['lr_se']:.4f}",
     "RF AUC": f"{b_rf:.4f} ± {b['rf_se']:.4f}",
     "Avg AUC": f"{b_avg:.4f}",
     "Status": "Null reference"},

    {"Stage": f"2. Ablation winner ({opt_name.split('.')[1].strip()})",
     "Raw": p_raw, "Dims": p_dims,
     "LR AUC": f"{p_lr:.4f} ± {p['lr_se']:.4f}",
     "RF AUC": f"{p_rf:.4f} ± {p['rf_se']:.4f}",
     "Avg AUC": f"{p_avg:.4f}",
     "Status": "Pre-LOFO hypothesis set"},

    {"Stage": f"3. Post-LOFO refined subset",
     "Raw": s_raw, "Dims": s_dims,
     "LR AUC": f"{s_lr:.4f} ± {s_lr_se:.4f}",
     "RF AUC": f"{s_rf:.4f} ± {s_rf_se:.4f}",
     "Avg AUC": f"{s_avg:.4f}",
     "Status": "Parsimonious optimum"},

    {"Stage": "Net lift / reduction (Δ vs. baseline)",
     "Raw":  f"{s_raw - b_raw:+d} ({(s_raw - b_raw) / b_raw:+.1%})",
     "Dims": f"{s_dims - b_dims:+d} ({(s_dims - b_dims) / b_dims:+.1%})",
     "LR AUC": f"{s_lr - b_lr:+.4f}",
     "RF AUC": f"{s_rf - b_rf:+.4f}",
     "Avg AUC": f"{s_avg - b_avg:+.4f}",
     "Status": "Parsimony gain"},
])
display(comp_df)

# Comparative Diagnostics & Empirical Progression
fig, axes = plt.subplots(1, 3, figsize=(17, 5), dpi=150)
x, w = np.arange(2), 0.26
stage_colors = [PALETTE[8], PALETTE[0], PALETTE[2]]

# Panel 1 — AUC by stage
ax1 = axes[0]
ax1.bar(x - w, [b_lr, b_rf], w,
        yerr=[b["lr_se"], b["rf_se"]], capsize=4,
        label="1. Baseline", color=stage_colors[0])
ax1.bar(x,     [p_lr, p_rf], w,
        yerr=[p["lr_se"], p["rf_se"]], capsize=4,
        label=f"2. {opt_name.split('.')[1].strip()}", color=stage_colors[1])
ax1.bar(x + w, [s_lr, s_rf], w,
        yerr=[s_lr_se, s_rf_se], capsize=4,
        label="3. Post-LOFO refined", color=stage_colors[2])
ax1.set(xticks=x, xticklabels=["Logistic Regression", "Random Forest"],
        ylim=(0.825, 0.865), ylabel="5-Fold CV ROC-AUC")
ax1.set_title("Cross-Validated ROC-AUC by Ablation Stage", weight="bold")
ax1.legend(loc="lower left", fontsize=8.5)

for i, vals in enumerate([[b_lr, p_lr, s_lr], [b_rf, p_rf, s_rf]]):
    for j, (v, c) in enumerate(zip(vals, stage_colors)):
        ax1.text(i + (j - 1) * w, v + 0.001, f"{v:.4f}",
                 ha="center", va="bottom", fontsize=8, weight="bold")

# delta AUC by stage, LR vs RF
ax2 = axes[1]
stages = ["Pre-LOFO\nvs Baseline", "Post-LOFO\nvs Pre-LOFO", "Total\nnet lift"]
x2, w2 = np.arange(len(stages)), 0.34

lr_gains = [p_lr - b_lr, s_lr - p_lr, s_lr - b_lr]
rf_gains = [p_rf - b_rf, s_rf - p_rf, s_rf - b_rf]
all_gains = lr_gains + rf_gains

bars_lr = ax2.bar(x2 - w2 / 2, lr_gains, w2,
                  label="Logistic Regression", color=PALETTE[0], alpha=0.9)
bars_rf = ax2.bar(x2 + w2 / 2, rf_gains, w2,
                  label="Random Forest", color=PALETTE[1], alpha=0.9)

ax2.axhline(0, color="black", lw=0.8, ls="--")
y_min = min(all_gains) * 1.4 if min(all_gains) < 0 else 0
y_max = max(all_gains) * 1.5 if max(all_gains) > 0 else 0.005
ax2.set(xticks=x2, xticklabels=stages, ylabel="Generalization lift (ΔAUC)",
        ylim=(y_min, y_max))
ax2.set_title("Empirical discrimination ΔAUC", weight="bold")
ax2.legend(loc="upper left", fontsize=8.5, framealpha=0.95)

for bars in [bars_lr, bars_rf]:
    for br in bars:
        h = br.get_height()
        va = "bottom" if h >= 0 else "top"
        off = 0.00008 if h >= 0 else -0.00004
        ax2.text(br.get_x() + br.get_width() / 2, h + off, f"{h:+.4f}",
                 ha="center", va=va, fontsize=8, weight="bold")

# parameter reduction
ax3 = axes[2]
ax3.bar(x - w, [b_raw, b_dims], w, label="Baseline", color=stage_colors[0])
ax3.bar(x,     [p_raw, p_dims], w, label=opt_name.split('.')[1].strip(),
        color=stage_colors[1])
ax3.bar(x + w, [s_raw, s_dims], w, label="Post-LOFO refined",
        color=stage_colors[2])
ax3.set(xticks=x, xticklabels=["Candidate predictors", "Encoded dimensions"],
        ylabel="Parameter count")
ax3.set_title(f"Parameter reduction (total: {(s_dims - b_dims) / b_dims:+.1%})",
              weight="bold")
ax3.legend(loc="upper left", fontsize=8.5)

for i, vals in enumerate([[b_raw, p_raw, s_raw], [b_dims, p_dims, s_dims]]):
    for j, (v, c) in enumerate(zip(vals, stage_colors)):
        diff = f" ({v - vals[0]:+d})" if j > 0 else ""
        ax3.text(i + (j - 1) * w, v + 0.6, f"{v}{diff}",
                 ha="center", va="bottom", fontsize=8, weight="bold")

for ax in axes:
    ax.grid(axis="y", ls=":", alpha=0.5)
plt.tight_layout()
plt.show()

---
## 8. Supervised Discretization & Weight of Evidence (WoE)

### Business Context
Customer behavior rarely changes in a strictly linear fashion. Churn risk often drops sharply after specific customer milestones (e.g., reaching 12 months of tenure or passing trial periods). Converting continuous variables into **behavioral risk bands** allows linear models to capture non-linear relationship patterns.

### Methodology
* **Mixed-Integer Linear Programming (MILP via OptBinning):** Finds optimal bin boundaries that maximize Information Value (IV) under monotonic risk constraints.
* **1-Standard-Error Rule:** Prunes bin counts to ensure each segment has enough customer volume for reliable commercial targeting.
* **Weight of Evidence (WoE) Encoding:** Converts categorical levels into the log-odds of churn, making non-linear dynamics accessible to linear scoring models.

In [ ]:
!pip install optbinning

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from optbinning import OptimalBinning
import cvxpy as cp


def get_dt_splits(s: pd.Series, y_target: pd.Series, k: int) -> np.ndarray:
    """Supervised bin edges for one continuous feature via a shallow decision tree."""
    df = pd.DataFrame({"x": s.values, "y": y_target.astype(int).values}).dropna()

    if k < 2 or df["x"].nunique() < 2 or df["y"].nunique() < 2:
        return np.array([-np.inf, np.inf])

    dt = DecisionTreeClassifier(
        max_leaf_nodes=int(k),
        min_samples_leaf=0.03,
        random_state=42,
    ).fit(df[["x"]], df["y"])

    is_internal = dt.tree_.children_left != dt.tree_.children_right
    thresholds  = dt.tree_.threshold[is_internal]
    return np.unique(np.concatenate([[-np.inf], np.sort(thresholds), [np.inf]]))


def calc_iv(binned_s: pd.Series, y_target: pd.Series, eps: float = 1e-6) -> float:
    """Information Value of a binned predictor against a binary target."""
    df = pd.DataFrame({
        "bin": binned_s.values,
        "y":   y_target.astype(int).values,
    }).dropna()

    if df["bin"].nunique() < 2 or df["y"].nunique() < 2:
        return 0.0

    ct = pd.crosstab(df["bin"], df["y"], normalize="columns")
    p1 = ct[1].clip(lower=eps)
    p0 = ct[0].clip(lower=eps)
    return float(((p1 - p0) * np.log(p1 / p0)).sum())


def get_optb_splits(s: pd.Series, y_target: pd.Series,
                    max_n_bins: int = 5,
                    min_bin_size: float = 0.03) -> np.ndarray:
    """Supervised bin edges from OptBinning (monotonic WoE binning)."""
    df = pd.DataFrame({"x": s.values, "y": y_target.astype(int).values}).dropna()

    if df["x"].nunique() < 2 or df["y"].nunique() < 2:
        return np.array([-np.inf, np.inf])

    optb = OptimalBinning(
        name="x", dtype="numerical",
        max_n_bins=max_n_bins, min_bin_size=min_bin_size,
        solver="cp",                    # cp | mip | ls,
    )
    optb.fit(df["x"], df["y"])
    edges = [-np.inf] + list(optb.splits) + [np.inf]
    return np.unique(np.array(edges, dtype=float))

In [ ]:
from sklearn.model_selection import StratifiedKFold
import cvxpy as cp

df_bin  = df_selected.reset_index(drop=True).copy()
y_train = df_bin[TARGET].astype(int).reset_index(drop=True)

NUMERIC_COLS = [
    c for c in df_bin.select_dtypes(include=np.number).columns
    if c != TARGET
    and not pd.api.types.is_bool_dtype(df_bin[c])
    and df_bin[c].nunique() > 2
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Optimal Binning & 1-SE Decision Tree Granularity Search
k_range   = list(range(2, 16))
k_records = []
optimal_k = {}
optb_k    = {}

for col in NUMERIC_COLS:
    # MILP-solver optimal bins learned strictly from y_train
    optb = OptimalBinning(
        name=col, dtype="numerical",
        solver="cp",                    # cp | mip | ls
        min_bin_size=0.05,
    ).fit(df_bin[col].values, y_train.values)
    optb_k[col] = len(optb.splits) + 1

    # Out-of-fold CV over k granularity within df_bin
    for k in k_range:
        aucs = []
        for tr, va in cv.split(df_bin, y_train):
            s_tr = df_bin[col].iloc[tr]
            s_va = df_bin[col].iloc[va]
            y_tr = y_train.iloc[tr]
            y_va = y_train.iloc[va]

            bins  = get_dt_splits(s_tr, y_tr, k)
            tr_b  = pd.cut(s_tr, bins=bins).astype(str)
            va_b  = pd.cut(s_va, bins=bins).astype(str)

            # Out-of-fold target encoding: bin → P(y=1 | bin, train)
            enc   = y_tr.groupby(tr_b).mean()
            preds = va_b.map(enc).fillna(y_tr.mean())

            aucs.append(roc_auc_score(y_va, preds))

        k_records.append({
            "col": col,
            "k":   k,
            "auc": float(np.mean(aucs)),
            "se":  float(np.std(aucs, ddof=1) / np.sqrt(len(aucs))),
        })

res_k_df = pd.DataFrame(k_records)

# 1-SE parsimonious optimal k per numeric column
for col in NUMERIC_COLS:
    sub  = res_k_df[res_k_df["col"] == col]
    best = sub.loc[sub["auc"].idxmax()]
    optimal_k[col] = int(
        sub.loc[sub["auc"] >= best["auc"] - best["se"], "k"].min()
    )

fig, axes = plt.subplots(
    2, len(NUMERIC_COLS),
    figsize=(5.4 * len(NUMERIC_COLS), 8),
    dpi=150,
    squeeze=False,                     # guarantees axes[i, j] indexing
)

for i, col in enumerate(NUMERIC_COLS):
    sub   = res_k_df[res_k_df["col"] == col].sort_values("k").reset_index(drop=True)
    k_v   = sub["k"].to_numpy()
    a_v   = sub["auc"].to_numpy()
    s_v   = sub["se"].to_numpy()
    k_opt = optimal_k[col]

    best_row  = sub.loc[sub["auc"].idxmax()]
    se_thresh = best_row["auc"] - best_row["se"]

    ax_t = axes[0, i]
    ax_t.plot(k_v, a_v, "o-", color=PALETTE[0], lw=2.2, label="5-Fold CV AUC")
    ax_t.fill_between(k_v, a_v - s_v, a_v + s_v,
                      color=PALETTE[0], alpha=0.15, label="±1 SE band")
    ax_t.axhline(se_thresh, color="gray", ls=":", lw=1.2,
                 label=f"1-SE cutoff ({se_thresh:.4f})")
    ax_t.axvline(k_opt, color=PALETTE[2], ls="--", lw=1.8,
                 label=f"1-SE optimal k* = {k_opt}")
    ax_t.set(title=f"{col}: out-of-fold AUC (df_selected)",
             ylabel="5-Fold CV ROC-AUC", xticks=k_v)
    ax_t.legend(loc="lower right", fontsize=8, framealpha=0.9)
    ax_t.grid(ls=":", alpha=0.5)

    ax_b = axes[1, i]
    gains  = np.diff(a_v) * 100
    k_gain = k_v[1:]
    colors = [PALETTE[2] if g >= 0 else PALETTE[1] for g in gains]

    ax_b.bar(k_gain, gains, color=colors, alpha=0.85,
             width=0.6, ec="black", lw=0.5)
    ax_b.axhline(0, color="black", lw=0.8)
    ax_b.axvline(k_opt, color=PALETTE[2], ls="--", lw=1.8)
    ax_b.set(xlabel="Bin granularity (k)",
             ylabel="Marginal ΔAUC (pp)",
             title=f"{col}: marginal gain (pp)",
             xticks=k_v)
    ax_b.grid(ls=":", alpha=0.5)

plt.tight_layout()
plt.show()

# summary
print("=" * 70)
print(f"Source frame         : df_selected "
      f"({len(df_bin):,} rows × {len(df_bin.columns)} cols)")
print(f"Numeric features     : {NUMERIC_COLS}")
print(f"DT 1-SE optimal bins : {optimal_k}")
print(f"OptBinning bins      : {optb_k}")
print("=" * 70)

### 8.1 Discretization Strategy Cross-Validation
* Compares five representation approaches: Raw Continuous, Equal-Width, 1-SE Decision Trees, MILP One-Hot, and Monotonic WoE.
* **Core Takeaway:** Linear scorecards benefit substantially from non-linear discretization, whereas tree-based ensembles perform best with dense, unbinned continuous inputs.

In [ ]:
from sklearn.metrics import f1_score
# Macro Discretization Strategy Cross-Validation Benchmark
# Reference frame: df_selected
df_strat = df_selected.reset_index(drop=True).copy()
y_strat  = df_strat[TARGET].astype(int).reset_index(drop=True)

# Numeric features present in df_selected — same rule as the binning cell
NUMERIC_COLS = [
    c for c in df_strat.select_dtypes(include=np.number).columns
    if c != TARGET
    and not pd.api.types.is_bool_dtype(df_strat[c])
    and df_strat[c].nunique() > 2
]

strategies = [
    ("Raw Continuous (No-binning)",  "Continuous",     {c: 1     for c in NUMERIC_COLS}),
    ("Equal-width (3 bins)",         "Coarse",         {c: 3     for c in NUMERIC_COLS}),
    ("1-SE Rule (Decision Tree)",    "Optimal Binned", optimal_k),
    ("One Hot (MILP Binning)",       "Optimal Binned", optb_k),
    ("Monotonic WoE (MILP Binning)", "WoE",            optb_k),
]

summary_records = []

for name, cat, k_map in strategies:
    lr_sc, lr_f1, rf_sc, rf_f1 = [], [], [], []
    total_iv = 0.0

    # In-sample Total IV (reporting only, on the full df_selected)
    for col in NUMERIC_COLS:
        if "MILP" in name:
            optb = OptimalBinning(
                name=col, dtype="numerical",
                solver="cp", min_bin_size=0.05,
            ).fit(df_strat[col].values, y_strat.values)
            total_iv += optb.binning_table.build()["IV"].values[-1]
        elif "Continuous" not in name:
            edges = get_dt_splits(df_strat[col], y_strat, k_map[col])
            total_iv += calc_iv(pd.cut(df_strat[col], bins=edges), y_strat)

    # 5-fold Stratified CV
    for tr, va in cv.split(df_strat, y_strat):
        df_tr, df_va = df_strat.iloc[tr], df_strat.iloc[va]
        y_tr,  y_va  = y_strat.iloc[tr],  y_strat.iloc[va]

        if name == "Raw Continuous (No-binning)":
            scl = StandardScaler()
            X_tr_lr = scl.fit_transform(df_tr[NUMERIC_COLS])
            X_va_lr = scl.transform(df_va[NUMERIC_COLS])
            X_tr_rf = df_tr[NUMERIC_COLS].to_numpy()
            X_va_rf = df_va[NUMERIC_COLS].to_numpy()

        elif name == "Monotonic WoE (MILP Binning)":
            X_tr_w = pd.DataFrame(index=df_tr.index)
            X_va_w = pd.DataFrame(index=df_va.index)
            for c in NUMERIC_COLS:
                optb = OptimalBinning(
                    name=c, dtype="numerical",
                    solver="cp", min_bin_size=0.05,
                ).fit(df_tr[c].values, y_tr.values)
                X_tr_w[c] = optb.transform(df_tr[c].values, metric="woe")
                X_va_w[c] = optb.transform(df_va[c].values, metric="woe")
            X_tr_lr = X_tr_rf = X_tr_w.to_numpy()
            X_va_lr = X_va_rf = X_va_w.to_numpy()

        # discrete bins -> one-hot
        else:
            X_tr_b = pd.DataFrame(index=df_tr.index)
            X_va_b = pd.DataFrame(index=df_va.index)
            for c in NUMERIC_COLS:
                if "MILP" in name:
                    optb = OptimalBinning(
                        name=c, dtype="numerical",
                        solver="cp", min_bin_size=0.05,
                    ).fit(df_tr[c].values, y_tr.values)
                    X_tr_b[c] = optb.transform(df_tr[c].values, metric="bins").astype(str)
                    X_va_b[c] = optb.transform(df_va[c].values, metric="bins").astype(str)
                else:
                    edges  = get_dt_splits(df_tr[c], y_tr, k_map[c])
                    X_tr_b[c] = pd.cut(df_tr[c], bins=edges).astype(str)
                    X_va_b[c] = pd.cut(df_va[c], bins=edges).astype(str)

            X_tr_ohe = pd.get_dummies(X_tr_b, drop_first=True)
            X_va_ohe = (
                pd.get_dummies(X_va_b, drop_first=True)
                  .reindex(columns=X_tr_ohe.columns, fill_value=0)
            )
            X_tr_lr = X_tr_rf = X_tr_ohe.to_numpy()
            X_va_lr = X_va_rf = X_va_ohe.to_numpy()

        # Fit + score ----
        lr = LogisticRegression(
            class_weight="balanced", max_iter=5000,
            random_state=42, n_jobs=-1,
        ).fit(X_tr_lr, y_tr)
        rf = RandomForestClassifier(
            n_estimators=100, max_depth=6, class_weight="balanced",
            random_state=42, n_jobs=-1,
        ).fit(X_tr_rf, y_tr)

        lr_sc.append(roc_auc_score(y_va, lr.predict_proba(X_va_lr)[:, 1]))
        lr_f1.append(f1_score(y_va, lr.predict(X_va_lr)))
        rf_sc.append(roc_auc_score(y_va, rf.predict_proba(X_va_rf)[:, 1]))
        rf_f1.append(f1_score(y_va, rf.predict(X_va_rf)))

    # Reported bin / dim counts
    if "Continuous" in name:
        total_bins = len(NUMERIC_COLS)
        enc_dims   = len(NUMERIC_COLS)
    elif "WoE" in name or "MILP" in name:
        total_bins = sum(k_map.values())
        enc_dims   = len(NUMERIC_COLS) if "WoE" in name else sum(k_map.values())
    else:
        total_bins = sum(k_map.values())
        # One-hot with drop_first: sum(k_i - 1) over features
        enc_dims   = sum(k_map[c] - 1 for c in NUMERIC_COLS)

    summary_records.append({
        "Strategy":     name,
        "Category":     cat,
        "Total Bins":   total_bins,
        "Encoded Dims": enc_dims,
        "Total IV":     f"{total_iv:.4f}" if total_iv > 0 else "N/A",
        "Monotonic":    "Yes" if "MILP" in name else "No",
        "LR CV AUC":    f"{np.mean(lr_sc):.4f} ± {np.std(lr_sc, ddof=1)/np.sqrt(len(lr_sc)):.4f}",
        "LR F1":        f"{np.mean(lr_f1):.4f}",
        "RF CV AUC":    f"{np.mean(rf_sc):.4f} ± {np.std(rf_sc, ddof=1)/np.sqrt(len(rf_sc)):.4f}",
        "RF F1":        f"{np.mean(rf_f1):.4f}",
        "lr_m":         np.mean(lr_sc),
        "lr_se":        np.std(lr_sc, ddof=1) / np.sqrt(len(lr_sc)),
        "rf_m":         np.mean(rf_sc),
        "rf_se":        np.std(rf_sc, ddof=1) / np.sqrt(len(rf_sc)),
    })

bench_df = pd.DataFrame(summary_records)

display(bench_df[["Strategy", "Category", "Total Bins", "Encoded Dims",
                  "Total IV", "Monotonic",
                  "LR CV AUC", "LR F1", "RF CV AUC", "RF F1"]])

# Visualization
fig, (ax_lr, ax_rf) = plt.subplots(1, 2, figsize=(16, 5.5), dpi=150)
strategy_colors = {
    "Raw Continuous (No-binning)":  PALETTE[0],
    "Equal-width (3 bins)":         PALETTE[3],
    "1-SE Rule (Decision Tree)":    PALETTE[2],
    "One Hot (MILP Binning)":       PALETTE[4],
    "Monotonic WoE (MILP Binning)": PALETTE[1],
}
bar_colors = [strategy_colors[s] for s in bench_df["Strategy"]]

y_min = min(bench_df["lr_m"].min(), bench_df["rf_m"].min()) - 0.015
y_max = max(bench_df["lr_m"].max(), bench_df["rf_m"].max()) + 0.015
x_pos, width = np.arange(len(bench_df)), 0.55

for ax, m_col, se_col, title in [
    (ax_lr, "lr_m", "lr_se", "Linear classifier (Logistic Regression)"),
    (ax_rf, "rf_m", "rf_se", "Tree ensemble (Random Forest)"),
]:
    bars = ax.bar(
        x_pos, bench_df[m_col], yerr=bench_df[se_col], capsize=4, width=width,
        color=bar_colors, alpha=0.88, ec="black", lw=0.8,
    )
    # Highlight best performer
    best_i = int(bench_df[m_col].idxmax())
    bars[best_i].set_edgecolor("#dc2626")
    bars[best_i].set_linewidth(2.2)

    ax.axhline(
        bench_df.loc[0, m_col], color=PALETTE[0], ls="--", alpha=0.6,
        label=f"Continuous baseline ({bench_df.loc[0, m_col]:.4f})",
    )
    ax.set(
        xticks=x_pos,
        xticklabels=[s.replace(" (", "\n(") for s in bench_df["Strategy"]],
        ylim=(y_min, y_max), ylabel="5-fold CV ROC-AUC",
    )
    ax.set_title(title, weight="bold", pad=10)
    ax.grid(axis="y", ls=":", alpha=0.5)
    ax.legend(loc="lower right", fontsize=8.5)

    for b, se in zip(bars, bench_df[se_col]):
        h = b.get_height()
        ax.annotate(
            f"{h:.4f}\n±{se:.4f}",
            xy=(b.get_x() + b.get_width() / 2, h + se),
            xytext=(0, 4), textcoords="offset points",
            ha="center", va="bottom", fontsize=8, weight="bold",
        )

plt.tight_layout()
plt.show()

In [ ]:
# ── Per-model optimal discretization strategy ──────────────
# Rationale: LR and RF don't agree, and they shouldn't.
#   - LR (linear in parameters): nonlinear bin transforms lift it.
#   - RF (tree splits): monotone-invariant, prefers dense low-dim input.

# Ensure the strategy → k-map lookup exists
if "strat_to_k" not in globals():
    NUMERIC_COLS = [
        c for c in df_selected.select_dtypes(include=np.number).columns
        if c != TARGET
        and not pd.api.types.is_bool_dtype(df_selected[c])
        and df_selected[c].nunique() > 2
    ]
    strat_to_k = {
        "Raw Continuous (No-binning)":  {c: 1 for c in NUMERIC_COLS},
        "Equal-width (3 bins)":         {c: 3 for c in NUMERIC_COLS},
        "1-SE Rule (Decision Tree)":    optimal_k,
        "One Hot (MILP Binning)":       optb_k,
        "Monotonic WoE (MILP Binning)": optb_k,
    }

# Attach k_map to bench_df — column lookup is more robust than dict lookup
bench_df["k_map"] = bench_df["Strategy"].map(strat_to_k)

lr_best = bench_df.loc[bench_df["lr_m"].idxmax()]
rf_best = bench_df.loc[bench_df["rf_m"].idxmax()]

lr_k_map = lr_best["k_map"]
rf_k_map = rf_best["k_map"]

print("=" * 78)
print("PER-MODEL OPTIMAL DISCRETIZATION STRATEGY")
print("=" * 78)
print(f"{'Model':<22} {'Strategy':<32} {'AUC':>14} {'Enc. Dims':>10}")
print("-" * 78)
print(f"{'Logistic Regression':<22} {lr_best['Strategy']:<32} "
      f"{lr_best['lr_m']:.4f} ± {lr_best['lr_se']:.4f} {int(lr_best['Encoded Dims']):>10}")
print(f"{'Random Forest':<22} {rf_best['Strategy']:<32} "
      f"{rf_best['rf_m']:.4f} ± {rf_best['rf_se']:.4f} {int(rf_best['Encoded Dims']):>10}")
print("-" * 78)

print()
print("Downstream recipes:")
print(f"  LR → {lr_best['Strategy']}")
print(f"       k_map: {lr_k_map}")
print(f"       {'continuous (no binning)' if 'Continuous' in lr_best['Strategy'] else 'requires binning'}")
print()
print(f"  RF → {rf_best['Strategy']}")
print(f"       k_map: {rf_k_map}")
print(f"       {'continuous (no binning)' if 'Continuous' in rf_best['Strategy'] else 'requires binning'}")

# Do the two models agree on the discretization?
agree = lr_best["Strategy"] == rf_best["Strategy"]
print()
if agree:
    print(f"  ✓ Both models prefer: {lr_best['Strategy']}")
else:
    print(f"  ✗ Models disagree — LR prefers {lr_best['Strategy']},")
    print(f"    RF prefers {rf_best['Strategy']}")

# Cross-check in AUC terms
print()
print("Cross-check: how each model fares on the OTHER's preferred strategy")
print("-" * 78)
for model_col, other_row, own_row, label in [
    ("lr_m", rf_best,  lr_best, "LR"),
    ("rf_m", lr_best,  rf_best, "RF"),
]:
    se_col    = model_col.replace("_m", "_se")
    own_auc   = own_row[model_col]
    other_auc = other_row[model_col]
    other_se  = other_row[se_col]
    delta     = own_auc - other_auc
    z_ish     = delta / other_se if other_se else 0
    print(f"  {label}: {own_auc:.4f} (own) vs {other_auc:.4f} (other) "
          f"→ Δ {delta:+.4f} ({z_ish:+.2f} SE)")
print("=" * 78)

# Persist for downstream cells
LR_STRATEGY = lr_best["Strategy"]
RF_STRATEGY = rf_best["Strategy"]
LR_K_MAP    = lr_k_map
RF_K_MAP    = rf_k_map

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder

# Inputs from the pruned frame
df_enc = df_selected.reset_index(drop=True).copy()
y      = df_enc[TARGET].astype(int).reset_index(drop=True)

NUMERIC_COLS = [
    c for c in df_enc.select_dtypes(include=np.number).columns
    if c != TARGET
    and not pd.api.types.is_bool_dtype(df_enc[c])
    and df_enc[c].nunique() > 2
]

# LR's preferred discretization (from the per-model strategy cell)
LR_STRATEGY = lr_best["Strategy"]
LR_K_MAP    = lr_best["k_map"]

print(f"LR discretization : {LR_STRATEGY}")
print(f"k_map             : {LR_K_MAP}")

def _edges_for(df_tr, y_tr, col, strategy, k):
    """Bin edges for one numeric feature, or None for raw continuous."""
    if "Continuous" in strategy:
        return None
    if "MILP" in strategy:
        optb = OptimalBinning(
            name=col, dtype="numerical", solver="cp", min_bin_size=0.05,
        ).fit(df_tr[col].values, y_tr.values)
        return np.unique(
            np.array([-np.inf] + list(optb.splits) + [np.inf], dtype=float)
        )
    return get_dt_splits(df_tr[col], y_tr, k)          # DT-based


def _woe_map(bins, y_tr, eps: float = 0.5) -> dict:
    """WoE per bin/category. Coerces keys to str so map() works regardless of input type."""
    bins = bins.astype(str)                  
    ct = pd.crosstab(bins, y_tr)
    for col in (0, 1):
        if col not in ct.columns:
            ct[col] = 0
    ct = ct + eps
    p_pos = ct[1] / ct[1].sum()
    p_neg = ct[0] / ct[0].sum()
    return np.log(p_pos / p_neg).to_dict()


#  Encoding benchmark (LR only) 
records = {"OHE": [], "WoE": [], "Ordinal": []}

for tr, va in cv.split(df_enc, y):
    df_tr, df_va = df_enc.iloc[tr], df_enc.iloc[va]
    y_tr,  y_va  = y.iloc[tr],  y.iloc[va]

    ohe_tr, ohe_va = {}, {}
    woe_tr, woe_va = {}, {}
    ord_tr, ord_va = {}, {}

    for c in NUMERIC_COLS:
        edges = _edges_for(df_tr, y_tr, c, LR_STRATEGY, LR_K_MAP[c])

        if edges is None:
            v_tr = df_tr[c].to_numpy()
            v_va = df_va[c].to_numpy()
            ohe_tr[c] = v_tr; ohe_va[c] = v_va
            woe_tr[c] = v_tr; woe_va[c] = v_va
            ord_tr[c] = v_tr; ord_va[c] = v_va
            continue

        cat_tr = pd.cut(df_tr[c], bins=edges)
        cat_va = pd.cut(df_va[c], bins=edges)

        # OHE
        ohe_tr[c] = cat_tr.astype(str)
        ohe_va[c] = cat_va.astype(str)

        # WoE
        wmap = _woe_map(cat_tr, y_tr)
        woe_tr[c] = cat_tr.astype(str).map(wmap).fillna(0).to_numpy()
        woe_va[c] = cat_va.astype(str).map(wmap).fillna(0).to_numpy()

        # Ordinal — .cat.codes = 0,1,2,... in bin order (NOT string sort)
        ord_tr[c] = cat_tr.cat.codes.replace(-1, 0).to_numpy()
        ord_va[c] = cat_va.cat.codes.replace(-1, 0).to_numpy()

    X_tr_ohe = pd.get_dummies(pd.DataFrame(ohe_tr), drop_first=True)
    X_va_ohe = (
        pd.get_dummies(pd.DataFrame(ohe_va), drop_first=True)
          .reindex(columns=X_tr_ohe.columns, fill_value=0)
    )
    X_tr_woe = pd.DataFrame(woe_tr)
    X_va_woe = pd.DataFrame(woe_va)
    X_tr_ord = pd.DataFrame(ord_tr)
    X_va_ord = pd.DataFrame(ord_va)

    for key, (xtr, xva) in [
        ("OHE",     (X_tr_ohe, X_va_ohe)),
        ("WoE",     (X_tr_woe, X_va_woe)),
        ("Ordinal", (X_tr_ord, X_va_ord)),
    ]:
        lr = LogisticRegression(
            class_weight="balanced", max_iter=5000, random_state=42,
        ).fit(xtr, y_tr)
        records[key].append(
            roc_auc_score(y_va, lr.predict_proba(xva)[:, 1])
        )


def _dims(key):
    if key == "OHE" and all(c in LR_K_MAP for c in NUMERIC_COLS):
        return sum(LR_K_MAP[c] - 1 for c in NUMERIC_COLS)
    return len(NUMERIC_COLS)


num_bench_df = pd.DataFrame([{
    "Numerical Bin Strategy": s,
    "Dimensions":  _dims(s),
    "LR 5-Fold ROC-AUC": f"{np.mean(sc):.4f} ± "
                          f"{np.std(sc, ddof=1)/np.sqrt(len(sc)):.4f}",
    "auc_mean": np.mean(sc),
    "auc_se":   np.std(sc, ddof=1) / np.sqrt(len(sc)),
} for s, sc in records.items()]).sort_values("auc_mean", ascending=False).reset_index(drop=True)

display(num_bench_df[["Numerical Bin Strategy", "Dimensions", "LR 5-Fold ROC-AUC"]])


# visualization
fig, ax = plt.subplots(figsize=(8.5, 5), dpi=150)

labels = [f"{s}\n({d} dims)"
          for s, d in zip(num_bench_df["Numerical Bin Strategy"],
                          num_bench_df["Dimensions"])]
x_pos  = np.arange(len(labels))
colors = [PALETTE[2] if i == 0 else PALETTE[8] for i in range(len(labels))]

bars = ax.bar(
    x_pos, num_bench_df["auc_mean"],
    yerr=num_bench_df["auc_se"], capsize=4,
    color=colors, alpha=0.88, edgecolor="black", linewidth=0.8, width=0.48,
)
bars[0].set_edgecolor("#dc2626")
bars[0].set_linewidth(2.0)

y_lo = (num_bench_df["auc_mean"] - num_bench_df["auc_se"]).min()
y_hi = (num_bench_df["auc_mean"] + num_bench_df["auc_se"]).max()
span = max(y_hi - y_lo, 0.01)
ax.set_ylim(y_lo - 0.25 * span, y_hi + 0.45 * span)

top_score = num_bench_df["auc_mean"].iloc[0]
ax.axhline(top_score, color=PALETTE[0], ls="--", lw=1.2, alpha=0.6,
           label=f"Selected optimum ({top_score:.4f})")

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=10, fontweight="bold")
ax.set_ylabel("5-Fold CV ROC-AUC (mean ± 1 SE)", fontsize=10.5, fontweight="bold")
ax.set_title(
    f"Numerical bin encoding — Logistic Regression\n"
    f"Discretization: {LR_STRATEGY}   |   OHE vs. WoE vs. Ordinal",
    fontsize=11.5, fontweight="bold", pad=12,
)
ax.grid(axis="y", linestyle=":", alpha=0.6)
ax.legend(loc="lower right", framealpha=0.95, fontsize=9)

for bar, se in zip(bars, num_bench_df["auc_se"]):
    h = bar.get_height()
    ax.annotate(
        f"{h:.4f}\n±{se:.4f}",
        xy=(bar.get_x() + bar.get_width() / 2, h + se),
        xytext=(0, 4), textcoords="offset points",
        ha="center", va="bottom", fontsize=8.5, fontweight="bold",
    )

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

df_enc = df_selected.reset_index(drop=True).copy()
y      = df_enc[TARGET].astype(int).reset_index(drop=True)

CATEGORICAL_COLS = [
    c for c in df_enc.select_dtypes(include=["object", "category"]).columns
    if c != TARGET
]

print(f"Categorical features ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")

def calc_woe_map(s: pd.Series, target: pd.Series, eps: float = 0.5) -> dict:
    """WoE per category. Add eps to every cell (symmetric smoothing)."""
    ct = pd.crosstab(s, target)
    for col in (0, 1):                       # both class columns must exist
        if col not in ct.columns:
            ct[col] = 0
    ct = ct + eps
    p_pos = ct[1] / ct[1].sum()
    p_neg = ct[0] / ct[0].sum()
    return np.log(p_pos / p_neg).to_dict()


cat_df = df_enc[CATEGORICAL_COLS]

cat_strategies = [
    ("Full OHE (drop_first=False)",
     OneHotEncoder(drop=None, sparse_output=False, handle_unknown="ignore")),
    ("Weight of Evidence (WoE)", "woe"),
    ("Ordinal Label Encoding",
     OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
]

cat_records = []

for name, enc in cat_strategies:
    lr_sc, dims = [], 0

    for tr, va in cv.split(cat_df, y):
        X_tr, X_va = cat_df.iloc[tr], cat_df.iloc[va]
        y_tr, y_va = y.iloc[tr], y.iloc[va]

        if name == "Weight of Evidence (WoE)":
            X_tr_e = pd.DataFrame(index=X_tr.index)
            X_va_e = pd.DataFrame(index=X_va.index)
            for c in CATEGORICAL_COLS:
                w_map = calc_woe_map(X_tr[c], y_tr)
                X_tr_e[c] = X_tr[c].map(w_map).fillna(0.0)
                X_va_e[c] = X_va[c].map(w_map).fillna(0.0)
        else:
            X_tr_e = enc.fit_transform(X_tr)
            X_va_e = enc.transform(X_va)

        dims = X_tr_e.shape[1]

        lr = LogisticRegression(
            class_weight="balanced", max_iter=5000, random_state=42,
        ).fit(X_tr_e, y_tr)
        lr_sc.append(roc_auc_score(y_va, lr.predict_proba(X_va_e)[:, 1]))

    lr_m  = float(np.mean(lr_sc))
    lr_se = float(np.std(lr_sc, ddof=1) / np.sqrt(len(lr_sc)))

    cat_records.append({
        "Categorical Encoding": name,
        "Dimensions":           dims,
        "LR 5-Fold ROC-AUC":    f"{lr_m:.4f} ± {lr_se:.4f}",
        "lr_m":                 lr_m,
        "lr_se":                lr_se,
    })

cat_bench_df = (
    pd.DataFrame(cat_records)
      .sort_values("lr_m", ascending=False)
      .reset_index(drop=True)
)

display(cat_bench_df[["Categorical Encoding", "Dimensions", "LR 5-Fold ROC-AUC"]])


fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

labels = [f"{s}\n({d} dims)"
          for s, d in zip(cat_bench_df["Categorical Encoding"],
                          cat_bench_df["Dimensions"])]
x_pos  = np.arange(len(labels))
colors = [PALETTE[2] if i == 0 else PALETTE[8] for i in range(len(labels))]

bars = ax.bar(
    x_pos, cat_bench_df["lr_m"],
    yerr=cat_bench_df["lr_se"], capsize=4,
    color=colors, alpha=0.88, edgecolor="black", linewidth=0.8, width=0.48,
)
bars[0].set_edgecolor("#dc2626")
bars[0].set_linewidth(2.0)

y_lo = (cat_bench_df["lr_m"] - cat_bench_df["lr_se"]).min()
y_hi = (cat_bench_df["lr_m"] + cat_bench_df["lr_se"]).max()
span = max(y_hi - y_lo, 0.01)
ax.set_ylim(y_lo - 0.25 * span, y_hi + 0.45 * span)

top_score = cat_bench_df["lr_m"].iloc[0]
ax.axhline(top_score, color=PALETTE[0], ls="--", lw=1.2, alpha=0.6,
           label=f"Selected optimum ({top_score:.4f})")

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=9.5, fontweight="bold")
ax.set_ylabel("5-Fold CV ROC-AUC (mean ± 1 SE)",
              fontsize=10.5, fontweight="bold")
ax.set_title(
    f"Categorical feature encoding — Logistic Regression\n"
    f"{len(CATEGORICAL_COLS)} categorical features   |   "
    f"OHE vs. WoE vs. Ordinal",
    fontsize=11.5, fontweight="bold", pad=12,
)
ax.grid(axis="y", linestyle=":", alpha=0.6)
ax.legend(loc="lower right", framealpha=0.95, fontsize=9)

for bar, se in zip(bars, cat_bench_df["lr_se"]):
    h = bar.get_height()
    ax.annotate(
        f"{h:.4f}\n±{se:.4f}",
        xy=(bar.get_x() + bar.get_width() / 2, h + se),
        xytext=(0, 4), textcoords="offset points",
        ha="center", va="bottom", fontsize=8.5, fontweight="bold",
    )

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy import stats

df_enc = df_selected.reset_index(drop=True).copy()
y      = df_enc[TARGET].astype(int).reset_index(drop=True)

NUMERIC_COLS     = [c for c in df_enc.select_dtypes(include=np.number).columns
                    if c != TARGET
                    and not pd.api.types.is_bool_dtype(df_enc[c])
                    and df_enc[c].nunique() > 2]
BOOLEAN_COLS     = [c for c in df_enc.select_dtypes("bool").columns if c != TARGET]
CATEGORICAL_COLS = [c for c in df_enc.select_dtypes(include=["object", "category"]).columns
                    if c != TARGET]

def _woe_map(s, y_tr, eps=0.5):
    s = s.astype(str)
    ct = pd.crosstab(s, y_tr)
    for col in (0, 1):
        if col not in ct.columns:
            ct[col] = 0
    ct = ct + eps
    return np.log((ct[1] / ct[1].sum()) / (ct[0] / ct[0].sum())).to_dict()

pipelines = [
    "Raw numeric + OHE cat",
    "Binned OHE numeric + OHE cat",
    "Binned OHE numeric + WoE cat",
]
pipe_records = {p: {"LR": [], "dims": 0} for p in pipelines}

for tr, va in cv.split(df_enc, y):
    df_tr, df_va = df_enc.iloc[tr], df_enc.iloc[va]
    y_tr,  y_va  = y.iloc[tr],  y.iloc[va]

    bool_tr = df_tr[BOOLEAN_COLS].astype(int)
    bool_va = df_va[BOOLEAN_COLS].astype(int)

    # Categorical OHE (drop_first) — reference pipelines
    ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
    cat_ohe_tr = pd.DataFrame(ohe.fit_transform(df_tr[CATEGORICAL_COLS]),
                              index=df_tr.index,
                              columns=ohe.get_feature_names_out(CATEGORICAL_COLS))
    cat_ohe_va = pd.DataFrame(ohe.transform(df_va[CATEGORICAL_COLS]),
                              index=df_va.index,
                              columns=ohe.get_feature_names_out(CATEGORICAL_COLS))

    # Categorical WoE (fold-local map)
    cat_woe_tr = pd.DataFrame(index=df_tr.index)
    cat_woe_va = pd.DataFrame(index=df_va.index)
    for c in CATEGORICAL_COLS:
        wmap = _woe_map(df_tr[c], y_tr)
        cat_woe_tr[c] = df_tr[c].astype(str).map(wmap).fillna(0.0)
        cat_woe_va[c] = df_va[c].astype(str).map(wmap).fillna(0.0)

    # Raw numerics (scaled)
    scl = StandardScaler()
    num_s_tr = pd.DataFrame(scl.fit_transform(df_tr[NUMERIC_COLS]),
                            index=df_tr.index, columns=NUMERIC_COLS)
    num_s_va = pd.DataFrame(scl.transform(df_va[NUMERIC_COLS]),
                            index=df_va.index, columns=NUMERIC_COLS)

    # Binned numerics -> OHE (uses the winning MILP discretization)
    n_b_tr, n_b_va = {}, {}
    for c in NUMERIC_COLS:
        edges = _edges_for(df_tr, y_tr, c, LR_STRATEGY, LR_K_MAP[c])
        n_b_tr[c] = pd.cut(df_tr[c], bins=edges).astype(str)
        n_b_va[c] = pd.cut(df_va[c], bins=edges).astype(str)

    num_ohe_tr = pd.get_dummies(pd.DataFrame(n_b_tr, index=df_tr.index), drop_first=True)
    num_ohe_va = (pd.get_dummies(pd.DataFrame(n_b_va, index=df_va.index), drop_first=True)
                    .reindex(columns=num_ohe_tr.columns, fill_value=0))

    X_tr = {
        "Raw numeric + OHE cat":        pd.concat([num_s_tr, cat_ohe_tr, bool_tr], axis=1),
        "Binned OHE numeric + OHE cat": pd.concat([num_ohe_tr, cat_ohe_tr, bool_tr], axis=1),
        "Binned OHE numeric + WoE cat": pd.concat([num_ohe_tr, cat_woe_tr, bool_tr], axis=1),
    }
    X_va = {
        "Raw numeric + OHE cat":        pd.concat([num_s_va, cat_ohe_va, bool_va], axis=1),
        "Binned OHE numeric + OHE cat": pd.concat([num_ohe_va, cat_ohe_va, bool_va], axis=1),
        "Binned OHE numeric + WoE cat": pd.concat([num_ohe_va, cat_woe_va, bool_va], axis=1),
    }

    for key in pipelines:
        lr = LogisticRegression(class_weight="balanced", max_iter=5000,
                                random_state=42).fit(X_tr[key], y_tr)
        pipe_records[key]["LR"].append(
            roc_auc_score(y_va, lr.predict_proba(X_va[key])[:, 1])
        )
        pipe_records[key]["dims"] = X_tr[key].shape[1]


# summary
full_pipe_df = pd.DataFrame([{
    "Pipeline":       p,
    "Dimensions":     pipe_records[p]["dims"],
    "LR 5-fold AUC":  f"{np.mean(pipe_records[p]['LR']):.4f} ± "
                      f"{np.std(pipe_records[p]['LR'], ddof=1)/np.sqrt(5):.4f}",
    "lr_mean":        float(np.mean(pipe_records[p]["LR"])),
    "lr_se":          float(np.std(pipe_records[p]["LR"], ddof=1) / np.sqrt(5)),
} for p in pipelines]).sort_values("lr_mean", ascending=False).reset_index(drop=True)

best_name  = full_pipe_df.iloc[0]["Pipeline"]
best_mean  = full_pipe_df.iloc[0]["lr_mean"]
best_folds = np.array(pipe_records[best_name]["LR"])

print(f"Winner (highest mean AUC) : {best_name}")
print(f"  Dimensions              : {full_pipe_df.iloc[0]['Dimensions']}")
print(f"  LR 5-fold AUC           : {best_mean:.4f}")
print()
print("Paired comparisons against the winner (same folds, same random_state):")
print("-" * 78)
print(f"  {'Pipeline':<38} {'Δ mean':>9} {'SE_diff':>9} {'t':>7} {'p':>7}")
print("-" * 78)
for p in pipelines:
    if p == best_name:
        continue
    folds   = np.array(pipe_records[p]["LR"])
    diffs   = best_folds - folds
    se_diff = diffs.std(ddof=1) / np.sqrt(len(diffs))
    t_stat, p_val = stats.ttest_rel(best_folds, folds)
    sig     = "***" if p_val < 0.001 else ("** " if p_val < 0.01 else ("*  " if p_val < 0.05 else "ns "))
    print(f"  {p:<38} {diffs.mean():>+9.4f} {se_diff:>9.4f} "
          f"{t_stat:>+7.2f} {p_val:>7.4f}  {sig}")
print("-" * 78)
print("Significance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant")

selected = full_pipe_df.iloc[0]
print()
print(f"Selected pipeline: {selected['Pipeline']} "
      f"({selected['Dimensions']} dims, AUC = {selected['lr_mean']:.4f})")

FINAL_PIPELINE = selected["Pipeline"]
FINAL_DIMS     = int(selected["Dimensions"])
FINAL_AUC      = float(selected["lr_mean"])

labels   = [f"{p}\n({d} dims)"
            for p, d in zip(full_pipe_df["Pipeline"], full_pipe_df["Dimensions"])]
means    = full_pipe_df["lr_mean"].to_numpy()
ses      = full_pipe_df["lr_se"].to_numpy()
x_pos    = np.arange(len(labels))

lo   = (means - ses).min();  hi = (means + ses).max()
span = max(hi - lo, 1e-4)
y_lo = lo - 0.25 * span
y_hi = hi + 0.50 * span

fig, ax = plt.subplots(figsize=(9.5, 5.2), dpi=150)
colors  = [PALETTE[2] if i == 0 else PALETTE[8] for i in range(len(labels))]

bars = ax.bar(x_pos, means, yerr=ses, capsize=4,
              color=colors, alpha=0.88, ec="black", lw=0.8, width=0.48)
bars[0].set_edgecolor("#dc2626")
bars[0].set_linewidth(2.0)

ax.axhline(means[0], color=PALETTE[0], ls="--", lw=1.2, alpha=0.6,
           label=f"Best ({means[0]:.4f})")

ax.set_ylim(y_lo, y_hi)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=9.5, fontweight="bold")
ax.set_ylabel("5-fold stratified CV ROC-AUC (mean ± 1 SE)",
              fontsize=10.5, fontweight="bold")
ax.set_title(
    "Full pipeline encoding benchmark — Logistic Regression\n"
    "Selected by highest mean AUC; paired tests shown in the console",
    fontsize=11.5, fontweight="bold", pad=12,
)
ax.grid(axis="y", linestyle=":", alpha=0.6)
ax.legend(loc="lower right", framealpha=0.95, fontsize=9)

for bar, se in zip(bars, ses):
    h = bar.get_height()
    ax.annotate(f"{h:.4f}\n±{se:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, h + se),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=8.5, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 9. Class Imbalance Remediation Assessment

### Business Context
Because churners make up ~26.5% of the portfolio, standard classification models can bias predictions toward non-churners. We test synthetic oversampling techniques (**SMOTE**, **ADASYN**, **SMOTE-ENN**) against baseline class-weight adjustments.

### Practical Consideration
Synthetic oversampling can improve raw recall, but it often distorts output probability calibration. When decisions rely on calculating exact expected revenue and costs, well-calibrated probabilities are critical.

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import ADASYN, SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy import stats

df_res = df_selected.reset_index(drop=True).copy()
y      = df_res[TARGET].astype(int).reset_index(drop=True)

NUMERIC_COLS     = [c for c in df_res.select_dtypes(include=np.number).columns
                    if c != TARGET
                    and not pd.api.types.is_bool_dtype(df_res[c])
                    and df_res[c].nunique() > 2]
BOOLEAN_COLS     = [c for c in df_res.select_dtypes("bool").columns if c != TARGET]
CATEGORICAL_COLS = [c for c in df_res.select_dtypes(include=["object", "category"]).columns
                    if c != TARGET]
NON_NUMERIC      = BOOLEAN_COLS + CATEGORICAL_COLS


# Tree matrix — raw continuous numerics + OHE categoricals, bools passthrough
X_tree = pd.get_dummies(df_res[[c for c in df_res.columns if c != TARGET]],
                        columns=CATEGORICAL_COLS, drop_first=True)
X_tree = X_tree.astype(float)

# Linear matrix — LR's winning numeric discretization + OHE categoricals + bools
binned_dict = {}
for c in NUMERIC_COLS:
    edges = _edges_for(df_res, y, c, LR_STRATEGY, LR_K_MAP[c])
    binned_dict[c] = pd.cut(df_res[c], bins=edges).astype(str)

X_num_bins = pd.get_dummies(pd.DataFrame(binned_dict), drop_first=True)
X_cat_ohe  = pd.get_dummies(df_res[CATEGORICAL_COLS], drop_first=True)
X_bool     = df_res[BOOLEAN_COLS].astype(float)

X_lin = pd.concat([X_num_bins, X_cat_ohe, X_bool], axis=1).astype(float)

print(f"Linear matrix : {X_lin.shape[1]:>3} dims  "
      f"|  numeric strategy: {LR_STRATEGY}")
print(f"Tree matrix   : {X_tree.shape[1]:>3} dims  "
      f"|  raw continuous numerics + OHE categoricals")


# Resampling strategies and model configurations
def _make_samplers(seed=42):
    """Fresh samplers per call — no state carried between folds."""
    return {
        "Baseline (None)":          None,
        "Class Weights (balanced)": None,
        "SMOTE":                    SMOTE(random_state=seed),
        "SMOTEENN":                 SMOTEENN(random_state=seed),
        "ADASYN":                   ADASYN(random_state=seed),
    }

STRATEGIES = ["Baseline (None)", "Class Weights (balanced)",
              "SMOTE", "SMOTEENN", "ADASYN"]

models = {
    "Linear (LogisticRegression)": {"data": X_lin,  "type": "linear"},
    "Tree (RandomForest)":         {"data": X_tree, "type": "tree"},
}

fold_results = {
    m: {s: {"roc": [], "pr": [], "rec": [], "prec": [], "f1": []}
        for s in STRATEGIES}
    for m in models
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Stratified 5-fold evaluation loop
for train_idx, val_idx in cv.split(X_tree, y):
    y_tr = y.iloc[train_idx]
    y_va = y.iloc[val_idx]

    for m_name, m_cfg in models.items():
        X_full = m_cfg["data"]
        X_tr = X_full.iloc[train_idx].copy().astype(float)
        X_va = X_full.iloc[val_idx].copy().astype(float)

        # Raw continuous numerics on the tree matrix — tree-internal scaling
        # is a no-op, but we scale anyway for parity with the linear path.
        cont_cols = [c for c in X_tr.columns
                     if pd.api.types.is_numeric_dtype(X_tr[c])
                     and X_tr[c].nunique() > 2]
        if cont_cols:
            scl = StandardScaler()
            X_tr[cont_cols] = scl.fit_transform(X_tr[cont_cols])
            X_va[cont_cols] = scl.transform(X_va[cont_cols])

        samplers = _make_samplers(seed=42)

        for s_name in STRATEGIES:
            sampler = samplers[s_name]
            cw      = "balanced" if s_name == "Class Weights (balanced)" else None

            # Resample strictly on the training fold
            if sampler is not None:
                X_tr_res, y_tr_res = sampler.fit_resample(X_tr, y_tr)
            else:
                X_tr_res, y_tr_res = X_tr, y_tr

            # Model
            if m_cfg["type"] == "linear":
                clf = LogisticRegression(
                    class_weight=cw, max_iter=5000, random_state=42,
                )
            else:
                clf = RandomForestClassifier(
                    n_estimators=100, max_depth=8,
                    class_weight=cw, random_state=42, n_jobs=-1,
                )

            clf.fit(X_tr_res, y_tr_res)

            y_proba = clf.predict_proba(X_va)[:, 1]
            y_pred  = clf.predict(X_va)

            fold_results[m_name][s_name]["roc"].append(
                roc_auc_score(y_va, y_proba))
            fold_results[m_name][s_name]["pr"].append(
                average_precision_score(y_va, y_proba))
            fold_results[m_name][s_name]["rec"].append(
                recall_score(y_va, y_pred, zero_division=0))
            fold_results[m_name][s_name]["prec"].append(
                precision_score(y_va, y_pred, zero_division=0))
            fold_results[m_name][s_name]["f1"].append(
                f1_score(y_va, y_pred, zero_division=0))


# Summary + paired audit against the baseline
MIN_EFFECT_ROC = 0.0020
MIN_EFFECT_PR  = 0.0030

summary_rows, audit_rows = [], []

for m_name in models:
    base_roc = np.array(fold_results[m_name]["Baseline (None)"]["roc"])
    base_pr  = np.array(fold_results[m_name]["Baseline (None)"]["pr"])
    b_roc_m, b_pr_m = base_roc.mean(), base_pr.mean()

    for s_name in STRATEGIES:
        res = fold_results[m_name][s_name]
        roc = np.array(res["roc"]);  pr = np.array(res["pr"])
        n   = len(roc)

        roc_m, roc_se = roc.mean(), roc.std(ddof=1) / np.sqrt(n)
        pr_m,  pr_se  = pr.mean(),  pr.std(ddof=1) / np.sqrt(n)
        rec_m, f1_m   = np.mean(res["rec"]), np.mean(res["f1"])

        summary_rows.append({
            "Architecture":        m_name,
            "Strategy":            s_name,
            "ROC-AUC (Mean ± SE)": f"{roc_m:.4f} ± {roc_se:.4f}",
            "PR-AUC (Mean ± SE)":  f"{pr_m:.4f} ± {pr_se:.4f}",
            "Recall":              f"{rec_m*100:.2f}%",
            "F1-Score":            f"{f1_m:.4f}",
            "roc_m": roc_m, "roc_se": roc_se,
            "pr_m": pr_m,  "pr_se": pr_se,
            "rec_m": rec_m, "f1_m": f1_m,
        })

        # Paired audit vs baseline
        if s_name == "Baseline (None)":
            d_roc = d_pr = 0.0
            se_d_roc = se_d_pr = 0.0
            decision = "Baseline reference (control)"
        else:
            d_roc = roc_m - b_roc_m
            d_pr  = pr_m  - b_pr_m
            se_d_roc = max(
                np.std(roc - base_roc, ddof=1) / np.sqrt(n),
                MIN_EFFECT_ROC,
            )
            se_d_pr  = max(
                np.std(pr - base_pr, ddof=1) / np.sqrt(n),
                MIN_EFFECT_PR,
            )

            roc_deg  = d_roc < -se_d_roc
            pr_deg   = d_pr  < -se_d_pr
            roc_lift = d_roc >  se_d_roc
            pr_lift  = d_pr  >  se_d_pr

            if roc_lift and pr_deg:
                decision = (f"Divergent trade-off: ROC lift ({d_roc:+.4f}) "
                            f"with PR degradation ({d_pr:+.4f})")
            elif pr_lift and roc_deg:
                decision = (f"Divergent trade-off: PR lift ({d_pr:+.4f}) "
                            f"with ROC degradation ({d_roc:+.4f})")
            elif roc_deg or pr_deg:
                decision = (f"Eliminate: observable degradation "
                            f"(ΔROC={d_roc:+.4f}, ΔPR={d_pr:+.4f})")
            elif roc_lift or pr_lift:
                decision = "Candidate: demonstrated performance lift"
            elif s_name == "Class Weights (balanced)":
                decision = "Retain: practically equivalent, preferred for imbalance"
            else:
                decision = "Eliminate: equivalent without lift (Occam's razor)"

        audit_rows.append({
            "Architecture":        m_name,
            "Candidate Strategy":  s_name,
            "Δ ROC-AUC":           f"{d_roc:+.4f}",
            "Δ PR-AUC":            f"{d_pr:+.4f}",
            "Protocol Evaluation": decision,
        })

resampling_bench_df = pd.DataFrame(summary_rows)
audit_df            = pd.DataFrame(audit_rows)

display(resampling_bench_df[[
    "Architecture", "Strategy",
    "ROC-AUC (Mean ± SE)", "PR-AUC (Mean ± SE)",
    "Recall", "F1-Score",
]])
display(audit_df)

---
## 10. Hyperparameter Tuning

We compare five machine learning architectures under an identical 5-Fold Stratified Cross-Validation framework:
1. **Logistic Regression:** Transparent, fully auditable, highly stable linear scorecard.
2. **Random Forest:** Non-parametric ensemble resilient to localized anomalies.
3. **XGBoost:** High-performance extreme gradient boosted trees.
4. **LightGBM:** Fast, leaf-wise gradient boosting optimized for split efficiency.
5. **CatBoost:** Optimized for categorical feature interactions.

Each model is evaluated across its hyperparameter space using out-of-fold ROC-AUC to balance discrimination power against operational simplicity.

In [ ]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

#  Inputs (positionally aligned; train-only frame) 
df_hp = df_selected.reset_index(drop=True).copy()
y_hp  = df_hp[TARGET].astype(int).reset_index(drop=True)

NUMERIC_COLS     = [c for c in df_hp.select_dtypes(include=np.number).columns
                    if c != TARGET
                    and not pd.api.types.is_bool_dtype(df_hp[c])
                    and df_hp[c].nunique() > 2]
BOOLEAN_COLS     = [c for c in df_hp.select_dtypes("bool").columns if c != TARGET]
CATEGORICAL_COLS = [c for c in df_hp.select_dtypes(include=["object", "category"]).columns
                    if c != TARGET]

print(f"Selected features — numeric: {len(NUMERIC_COLS)} | bool: {len(BOOLEAN_COLS)} "
      f"| categorical: {len(CATEGORICAL_COLS)}")
print(f"LR numeric discretization: {LR_STRATEGY}")


#  WoE helper (fold-local usage) 
def _woe_map(s, y_tr, eps=0.5):
    s = s.astype(str)
    ct = pd.crosstab(s, y_tr)
    for col in (0, 1):
        if col not in ct.columns:
            ct[col] = 0
    ct = ct + eps
    return np.log((ct[1] / ct[1].sum()) / (ct[0] / ct[0].sum())).to_dict()



# 1. LINEAR representation  —  LR's winning recipe
#    numeric: OHE on MILP bins     categorical: WoE     bool: passthrough

tr_binned = {}
for c in NUMERIC_COLS:
    edges = _edges_for(df_hp, y_hp, c, LR_STRATEGY, LR_K_MAP[c])
    tr_binned[c] = pd.cut(df_hp[c], bins=edges).astype(str)

X_num_bins = pd.get_dummies(pd.DataFrame(tr_binned), drop_first=True)

# WoE for categoricals — fit on the FULL train frame here ONLY to build the
# matrix for hyperparameter search. The CV inside GridSearchCV will use these
# WoE values as fixed columns; the search is not re-encoding per fold. This is
# a *mild* optimism (category statistics seen across all folds), but it does
# NOT leak from df_test. Strictly fold-local encoding is done later when
# building the final fitted model.
X_cat_woe = pd.DataFrame(index=df_hp.index)
for c in CATEGORICAL_COLS:
    wmap = _woe_map(df_hp[c], y_hp)
    X_cat_woe[c] = df_hp[c].astype(str).map(wmap).fillna(0.0)

X_bool_lin = df_hp[BOOLEAN_COLS].astype(float)

X_lin_hp = pd.concat([X_num_bins, X_cat_woe, X_bool_lin], axis=1).astype(float)



# 2. TREE representation  —  RF / boosted trees
#    numeric: raw continuous      categorical: OHE      bool: passthrough

X_tree_hp = (
    pd.get_dummies(df_hp.drop(columns=[TARGET]),
                   columns=CATEGORICAL_COLS, drop_first=True)
      .astype(float)
)

print(f"Linear dims: {X_lin_hp.shape[1]} | Tree dims: {X_tree_hp.shape[1]}")



# 3. Grid search

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 76)
print("HYPERPARAMETER GRID SEARCH — 5-fold stratified CV on df_selected (train only)")
print("=" * 76)


# 1 — Logistic Regression  (linear representation)
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=5000, random_state=42),
    {"C": [0.01, 0.1, 1.0, 5.0, 10.0]},
    cv=cv, scoring="roc_auc", n_jobs=-1,
).fit(X_lin_hp, y_hp)
print(f"  LogisticRegression : {grid_lr.best_score_:.4f} | {grid_lr.best_params_}")


# 2 — Random Forest  (tree representation: raw continuous numerics + OHE cat)
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1),
    {"n_estimators":     [100, 200],
     "max_depth":        [6, 8, 12],
     "min_samples_leaf": [5, 10]},
    cv=cv, scoring="roc_auc", n_jobs=-1,
).fit(X_tree_hp, y_hp)
print(f"  RandomForest       : {grid_rf.best_score_:.4f} | {grid_rf.best_params_}")


# 3 — CatBoost  (tree representation)
grid_cat = GridSearchCV(
    CatBoostClassifier(
        iterations=250, loss_function="Logloss",
        verbose=False, random_seed=42, thread_count=1,
    ),
    {"depth":         [3, 4],
     "learning_rate": [0.03, 0.05],
     "l2_leaf_reg":   [3, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1,
).fit(X_tree_hp, y_hp)
print(f"  CatBoost           : {grid_cat.best_score_:.4f} | {grid_cat.best_params_}")


# 4 — XGBoost  (tree representation)
grid_xgb = GridSearchCV(
    XGBClassifier(
        n_estimators=150, random_state=42,
        eval_metric="logloss", tree_method="hist", n_jobs=1,
    ),
    {"max_depth":     [3, 4],
     "learning_rate": [0.02, 0.04],
     "subsample":     [0.8, 1.0]},
    cv=cv, scoring="roc_auc", n_jobs=-1,
).fit(X_tree_hp, y_hp)
print(f"  XGBoost            : {grid_xgb.best_score_:.4f} | {grid_xgb.best_params_}")


# 5 — LightGBM  (tree representation)
grid_lgb = GridSearchCV(
    LGBMClassifier(
        n_estimators=150, random_state=42,
        verbose=-1, n_jobs=1,          # avoid OpenMP deadlock under GridSearchCV
    ),
    {"max_depth":     [3, 4],
     "num_leaves":    [7, 15],
     "learning_rate": [0.02, 0.04]},
    cv=cv, scoring="roc_auc", n_jobs=-1,
).fit(X_tree_hp, y_hp)
print(f"  LightGBM           : {grid_lgb.best_score_:.4f} | {grid_lgb.best_params_}")

print("=" * 76)


#  Bundle best params + CV AUCs for downstream use 
best_params = {
    "LogisticRegression": grid_lr.best_params_,
    "RandomForest":       grid_rf.best_params_,
    "CatBoost":           grid_cat.best_params_,
    "XGBoost":            grid_xgb.best_params_,
    "LightGBM":           grid_lgb.best_params_,
}
best_cv_auc = {
    "LogisticRegression": grid_lr.best_score_,
    "RandomForest":       grid_rf.best_score_,
    "CatBoost":           grid_cat.best_score_,
    "XGBoost":            grid_xgb.best_score_,
    "LightGBM":           grid_lgb.best_score_,
}

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
import pandas as pd

#  1. Derive TOP-N from the grid-search output 
TOP_N  = 3
ranked = sorted(best_cv_auc.items(), key=lambda kv: kv[1], reverse=True)
TOP3   = [name for name, _ in ranked[:TOP_N]]

FITTED_GRIDS = {
    "LogisticRegression": grid_lr,
    "RandomForest":       grid_rf,
    "CatBoost":           grid_cat,
    "XGBoost":            grid_xgb,
    "LightGBM":           grid_lgb,
}
MATRICES = {
    "LogisticRegression": X_lin_hp,
    "RandomForest":       X_tree_hp,
    "CatBoost":           X_tree_hp,
    "XGBoost":            X_tree_hp,
    "LightGBM":           X_tree_hp,
}

print(f"TOP{TOP_N} models by CV AUC:")
for i, (name, auc) in enumerate(ranked[:TOP_N], 1):
    print(f"  {i}. {name:<22} {auc:.4f}")

#  2. Per-fold OOF predictions (same cv object as grid search) 
oof = {}
for name in TOP3:
    clf   = FITTED_GRIDS[name].best_estimator_
    X_mat = MATRICES[name]
    oof[name] = cross_val_predict(
        clf, X_mat, y_hp,
        cv=cv, method="predict_proba", n_jobs=-1,
    )[:, 1]

#  3. Correlation matrix of OOF probabilities 
corr = pd.DataFrame(oof).corr()
print("\nOOF prediction correlation matrix:")
print(corr.round(4))
print()

#  4. Individual OOF AUCs 
print("Individual OOF AUCs:")
for name, p in oof.items():
    print(f"  {name:<22} {roc_auc_score(y_hp, p):.4f}")
print()

#  5. Soft-vote variants (all pairs and the full triple) 
from itertools import combinations
print("Soft-vote variants (equal weights, on OOF):")
for r in range(2, len(TOP3) + 1):
    for members in combinations(TOP3, r):
        avg = np.mean([oof[m] for m in members], axis=0)
        label = " + ".join(members)
        print(f"  {label:<48} {roc_auc_score(y_hp, avg):.4f}")

---
## 11. Pipeline Freezing & Scoring

A high-performing model is ineffective if its data transformations cannot be reliably reproduced in production. Here, we encapsulate all training parameters—binning thresholds, WoE lookups, and scaling factors—into an immutable, reproducible scoring pipeline ready for batch scoring or live API deployment.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, ConfusionMatrixDisplay,
)

#  1. Top model from the grid search 
top_name = ranked[0][0]              # e.g. "LogisticRegression"
top_auc  = ranked[0][1]
top_grid = FITTED_GRIDS[top_name]

print(f"Top model : {top_name}")
print(f"CV AUC    : {top_auc:.4f}")


#  2. Freeze encodings on df_selected (train-only) 
# These transforms are what make the test matrix comparable to the
# train matrix. Every parameter is learned on df_selected only.
frozen = {}

# Numeric bin edges — from LR's winning discretization
edges_by_col = {}
for c in NUMERIC_COLS:
    edges_by_col[c] = _edges_for(df_hp, y_hp, c, LR_STRATEGY, LR_K_MAP[c])
frozen["edges"] = edges_by_col

# WoE maps for categoricals
woe_maps = {}
for c in CATEGORICAL_COLS:
    woe_maps[c] = _woe_map(df_hp[c], y_hp)
frozen["woe"] = woe_maps


def build_linear_frame(df_in: pd.DataFrame) -> pd.DataFrame:
    """Apply the frozen encodings to any dataframe with the same schema."""
    # Numeric → OHE on the frozen bins
    binned = {
        c: pd.cut(df_in[c], bins=frozen["edges"][c]).astype(str)
        for c in NUMERIC_COLS
    }
    X_num = pd.get_dummies(pd.DataFrame(binned), drop_first=True)

    # Categorical → WoE (unseen categories → 0)
    X_cat = pd.DataFrame(index=df_in.index)
    for c in CATEGORICAL_COLS:
        X_cat[c] = df_in[c].astype(str).map(frozen["woe"][c]).fillna(0.0)

    # Booleans → passthrough float
    X_bool = df_in[BOOLEAN_COLS].astype(float)

    X = pd.concat([X_num, X_cat, X_bool], axis=1).astype(float)
    # Align to the training column set (fills any fold-missing dummies with 0)
    return X.reindex(columns=X_lin_hp.columns, fill_value=0.0)


def build_tree_frame(df_in: pd.DataFrame) -> pd.DataFrame:
    """Tree representation: raw numerics + OHE categoricals + booleans."""
    X = pd.get_dummies(df_in.drop(columns=[TARGET]),
                       columns=CATEGORICAL_COLS, drop_first=True).astype(float)
    return X.reindex(columns=X_tree_hp.columns, fill_value=0.0)


#  3. Fit the top model on the full train set 
# df_hp and y_hp are the train-only frames from the hyperparameter cell.
if top_name == "LogisticRegression":
    X_train_final = X_lin_hp
    X_test_final  = build_linear_frame(df_test.reset_index(drop=True))
else:
    X_train_final = X_tree_hp
    X_test_final  = build_tree_frame(df_test.reset_index(drop=True))

y_train_final = y_hp
y_test_final  = df_test[TARGET].astype(int).reset_index(drop=True)

final_clf = top_grid.best_estimator_
final_clf.fit(X_train_final, y_train_final)

#  4. Predict on the untouched test set 
probas = final_clf.predict_proba(X_test_final)[:, 1]
preds  = (probas >= 0.50).astype(int)          # default threshold
y_true = y_test_final.values


#  5. Metrics 
tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

eval_summary = {
    "Pipeline Suite":         "Final Selected Pipeline",
    "Model Architecture":     top_name,
    "Test Accuracy":          round(accuracy_score(y_true, preds), 4),
    "Test Precision":         round(precision_score(y_true, preds, zero_division=0), 4),
    "Test Recall":            round(recall_score(y_true, preds, zero_division=0), 4),
    "Test F1-Score":          round(f1_score(y_true, preds, zero_division=0), 4),
    "Test ROC-AUC":           round(roc_auc_score(y_true, probas), 4),
    "Test PR-AUC":            round(average_precision_score(y_true, probas), 4),
    "Brier Score":            round(brier_score_loss(y_true, probas), 4),
    "False Negatives (FN)":   int(fn),
    "False Positives (FP)":   int(fp),
}
display(pd.DataFrame([eval_summary]))

#  6. Confusion matrix plot 
fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_true, preds),
    display_labels=["Retained (0)", "Churn (1)"],
)
disp.plot(cmap="Blues", values_format="d", ax=ax, colorbar=False)
ax.set_title(f"Confusion Matrix — {top_name} (threshold = 0.50)",
             weight="bold", fontsize=12, pad=12)
ax.grid(False)
plt.tight_layout()
plt.show()

#  7. Persist for downstream threshold tuning 
final_model = {
    "name":         top_name,
    "estimator":    final_clf,
    "y_true":       y_true,
    "y_pred_proba": probas,
    "y_pred":       preds,
    "X_test":       X_test_final,
}
print(f"\nfinal_model dict populated for {top_name}")
print(f"Test rows: {len(y_true)} | Test positives: {int(y_true.sum())}")

---
## 12. Statistical Diagnostics & Probability Calibration Audit

For commercial retention campaigns, a model's predicted probability must reflect real-world event frequencies. If an algorithm assigns an account a 30% churn risk, roughly 30 out of 100 similar accounts should actually cancel.

### Diagnostic Panels
1. **Probability Separation:** Visualizes how cleanly the model separates churning accounts from retained subscribers.
2. **Discrimination Curves (ROC & Precision-Recall):** Confirms ranking stability across varying false-alarm tolerances.
3. **Calibration Curve:** Validates whether predicted probabilities match observed customer churn rates across all deciles.

In [ ]:
# Block 7.6: Complete Model Diagnostic Dashboard
from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    roc_auc_score, average_precision_score, brier_score_loss,
)
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#  1. Retrieve predictions from the final_model dict 
y_true     = np.asarray(final_model["y_true"])
y_prob     = np.asarray(final_model["y_pred_proba"])
model_name = final_model["name"]
prevalence = y_true.mean()

# Tuned threshold if you've set one; otherwise 0.50
thr = final_model.get("threshold", 0.50)

#  2. 2×2 diagnostic canvas 
fig_diag, axes_diag = plt.subplots(2, 2, figsize=(14, 11), dpi=150)
primary_color = "#2b5c8f"

# Panel 1 — Probability distribution by ground truth
ax1 = axes_diag[0, 0]
sns.histplot(y_prob[y_true == 0], bins=25, color="#10b981", alpha=0.45,
             label="Retained (Class 0)", ax=ax1, kde=True, stat="density")
sns.histplot(y_prob[y_true == 1], bins=25, color="#d95f02", alpha=0.45,
             label="Churn (Class 1)", ax=ax1, kde=True, stat="density")
ax1.axvline(0.50,      color="black", ls=":",  lw=2.0,
            label="Default cutoff (t = 0.50)")
ax1.axvline(thr,       color="#dc2626", ls="-", lw=2.0,
            label=f"Chosen cutoff (t = {thr:.2f})")
ax1.axvline(prevalence, color="gray", ls="--", lw=1.5,
            label=f"True base rate ({prevalence:.1%})")
ax1.set_title("Predicted probability distribution by ground truth",
              fontsize=11.5, weight="bold")
ax1.set_xlabel("Predicted churn probability", weight="bold")
ax1.set_ylabel("Density", weight="bold")
ax1.legend(loc="upper right", frameon=True, fontsize=9)
ax1.grid(True, ls=":", alpha=0.5)

# Panel 2 — ROC curve
ax2 = axes_diag[0, 1]
fpr, tpr, _ = roc_curve(y_true, y_prob)
auc_val = roc_auc_score(y_true, y_prob)
ax2.plot(fpr, tpr, color=primary_color, lw=2.5,
         label=f"{model_name} (ROC-AUC = {auc_val:.4f})")
ax2.plot([0, 1], [0, 1], "k:", lw=1.2, label="Random chance (0.5000)")
ax2.set_title("Receiver Operating Characteristic (ROC) curve",
              fontsize=11.5, weight="bold")
ax2.set_xlabel("False positive rate (1 − specificity)", weight="bold")
ax2.set_ylabel("True positive rate (recall)", weight="bold")
ax2.legend(loc="lower right", frameon=True, fontsize=9)
ax2.grid(True, ls=":", alpha=0.5)

# Panel 3 — Precision-recall curve
ax3 = axes_diag[1, 0]
prec, rec, _ = precision_recall_curve(y_true, y_prob)
pr_auc_val = average_precision_score(y_true, y_prob)
ax3.plot(rec, prec, color=primary_color, lw=2.5,
         label=f"{model_name} (PR-AUC = {pr_auc_val:.4f})")
ax3.axhline(prevalence, color="k", ls=":", lw=1.2,
            label=f"Base prevalence ({prevalence:.1%})")
ax3.set_title("Precision-recall (PR) curve", fontsize=11.5, weight="bold")
ax3.set_xlabel("Recall (sensitivity)", weight="bold")
ax3.set_ylabel("Precision (PPV)", weight="bold")
ax3.legend(loc="upper right", frameon=True, fontsize=9)
ax3.grid(True, ls=":", alpha=0.5)

# Panel 4 — Calibration / reliability diagram
ax4 = axes_diag[1, 1]
prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10,
                                          strategy="uniform")
brier_val = brier_score_loss(y_true, y_prob)
ax4.plot(prob_pred, prob_true, "s-", color=primary_color, lw=2.2,
         label=f"{model_name} (Brier = {brier_val:.4f})")
ax4.plot([0, 1], [0, 1], "k--", lw=1.2, label="Perfect calibration")
ax4.set_title("Probability calibration (reliability curve)",
              fontsize=11.5, weight="bold")
ax4.set_xlabel("Mean predicted probability (10 bins)", weight="bold")
ax4.set_ylabel("Empirical fraction of positives", weight="bold")
ax4.legend(loc="lower right", frameon=True, fontsize=9)
ax4.grid(True, ls=":", alpha=0.5)

plt.suptitle(f"Diagnostic dashboard — {model_name}",
             fontsize=14, weight="bold", y=1.01)
plt.tight_layout()
plt.show()

#  Summary print 
print(f"Model             : {model_name}")
print(f"Test rows         : {len(y_true):,}")
print(f"Class prevalence  : {prevalence:.4f}")
print(f"ROC-AUC           : {auc_val:.4f}")
print(f"PR-AUC            : {pr_auc_val:.4f}")
print(f"Brier score       : {brier_val:.4f}  (lower is better; "
      f"0.25 = trivial, ~0.14 = good)")
print(f"Chosen cutoff     : {thr:.2f}")

---
## 13. Financial Optimization: Cost-Aware Decision Strategy

### Executive Cost Matrix
Standard analytics pipelines assign binary predictions at an arbitrary 0.50 probability cutoff. In business operations, this ignores the asymmetry in financial outcomes:

| Scenario | Classification | Commercial Impact | Assigned Cost |
| :--- | :--- | :--- | :--- |
| **Missed Churner** | False Negative (FN) | Churner leaves uncontacted; unrecovered Lifetime Value loss | **$C_{\text{FN}} = \$700$** |
| **Unnecessary Offer** | False Positive (FP) | Loyal customer contacted; wasted promotional concession | **$C_{\text{FP}} = \$150$** |
| **Successful Save** | True Positive (TP) | Churn prevented; customer retained via incentive | Operational Win |
| **Untouched Loyal** | True Negative (TN) | Organic renewal; zero budget spent | Natural Baseline |

### Theoretical Break-Even Point
The optimal probability threshold where the expected cost of taking action matches the cost of inaction is calculated as:

$$p_{\text{breakeven}} = \frac{C_{\text{FP}}}{C_{\text{FN}} + C_{\text{FP}}} = \frac{150}{700 + 150} \approx 17.6\%$$

**Executive Decision Rule:** Any account with a predicted churn probability above **~17.6%** justifies an immediate retention incentive, because the expected cost of inaction outweighs the cost of the intervention.

In [ ]:
# Business Insight: Cost-Aware Retention Policy Comparison (default t = 0.50)
from sklearn.metrics import confusion_matrix

#  Economic parameters 
C_FN = 700.0        # lost CLV when a churner is missed
C_FP = 150.0        # retention-offer cost per targeted non-churner
p_breakeven = C_FP / (C_FN + C_FP)

#  Test set + top-model predictions at t = 0.50 
y_test_arr = np.asarray(final_model["y_true"])
p_test     = np.asarray(final_model["y_pred_proba"])
y_pred_50  = (p_test >= 0.50).astype(int)
n_test     = len(y_test_arr)
n_churners = int(y_test_arr.sum())
n_retained = n_test - n_churners

#  Policy builders 
def _policy(name, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_test_arr, y_pred).ravel()
    return {
        "policy":    name,
        "n_target":  int(tp + fp),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "recall":    tp / (tp + fn) if (tp + fn) else 0.0,
        "precision": tp / (tp + fp) if (tp + fp) else 0.0,
        "cost":      float(fn * C_FN + fp * C_FP),
    }

policies = [
    _policy("1. Null action",                 np.zeros_like(y_test_arr)),
    _policy("2. Universal concession",        np.ones_like(y_test_arr)),
    _policy("3. Model @ t = 0.50",            y_pred_50),
]
null_cost = policies[0]["cost"]

#  Bootstrap CIs (resample test set) 
n_boot, rng = 1000, np.random.default_rng(42)
boot = {p["policy"]: [] for p in policies}

for _ in range(n_boot):
    idx = rng.choice(n_test, size=n_test, replace=True)
    y_b = y_test_arr[idx]
    for p, y_pred_b in zip(policies, [np.zeros_like(y_b),
                                       np.ones_like(y_b),
                                       (p_test[idx] >= 0.50).astype(int)]):
        _, fp_b, fn_b, _ = confusion_matrix(y_b, y_pred_b).ravel()
        boot[p["policy"]].append(float(fn_b * C_FN + fp_b * C_FP))

#  Comparison table 
rows = []
for p in policies:
    b     = np.array(boot[p["policy"]])
    saved = null_cost - p["cost"]
    rows.append({
        "Policy":             p["policy"],
        "Targeted":           p["n_target"],
        "TP":                 p["tp"],
        "FN":                 p["fn"],
        "FP":                 p["fp"],
        "Recall":             f"{p['recall']*100:.1f}%",
        "Precision":          f"{p['precision']*100:.1f}%",
        "Total cost":         f"${p['cost']:,.0f}",
        "Cost (mean ± SD)":   f"${b.mean():,.0f} ± ${b.std():,.0f}",
        "95% CI":             f"[${np.percentile(b, 2.5):,.0f}, "
                              f"${np.percentile(b, 97.5):,.0f}]",
        "Net saved vs. null": f"${saved:,.0f}",
        "Reduction":          f"{saved / null_cost * 100:.1f}%",
    })
business_df = pd.DataFrame(rows).set_index("Policy")
display(business_df)

labels       = ["1. Null action\n(zero intervention)",
                "2. Universal concession\n(blanket outreach)",
                "3. Model @ t = 0.50\n(default cutoff)"]
exact_costs  = [p["cost"] for p in policies]
sds          = [float(np.std(boot[p["policy"]])) for p in policies]
fn_breakdown = [p["fn"] * C_FN for p in policies]
fp_breakdown = [p["fp"] * C_FP for p in policies]

COLOR_FN, COLOR_FP = "#c0392b", "#f39c12"
y_max = max(np.array(exact_costs) + np.array(sds)) * 1.55

fig, ax = plt.subplots(figsize=(11, 7), dpi=180)
ax.bar(labels, fn_breakdown, width=0.55, color=COLOR_FN,
       edgecolor="black", linewidth=1.5,
       label=f"Silent churn loss  (FN × ${C_FN:,.0f})")
ax.bar(labels, fp_breakdown, bottom=fn_breakdown, width=0.55, color=COLOR_FP,
       edgecolor="black", linewidth=1.5,
       label=f"Wasted concession spend  (FP × ${C_FP:,.0f})")
ax.errorbar(labels, exact_costs, yerr=sds, fmt="none", ecolor="black",
            capsize=8, capthick=2, elinewidth=2, alpha=0.85, zorder=10)

for i, (cost, sd) in enumerate(zip(exact_costs, sds)):
    reduction = (null_cost - cost) / null_cost * 100
    header = (f"${cost:,.0f}\n±${sd:,.0f}\n[null reference]" if i == 0
              else f"${cost:,.0f}\n±${sd:,.0f}\n[{reduction:.1f}% vs null]")
    ax.text(i, cost + sd + y_max * 0.02, header,
            ha="center", va="bottom", fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", fc="#f8f9fa", ec="#343a40"))

    fn_v, fp_v = fn_breakdown[i], fp_breakdown[i]
    if fn_v > cost * 0.10:
        ax.text(i, fn_v / 2, f"FN\n${fn_v/1000:.0f}k\n({fn_v/cost:.0%})",
                ha="center", va="center", color="white",
                fontsize=9.5, fontweight="bold")
    if fp_v > cost * 0.10:
        ax.text(i, fn_v + fp_v / 2, f"FP\n${fp_v/1000:.0f}k\n({fp_v/cost:.0%})",
                ha="center", va="center", color="white",
                fontsize=9.5, fontweight="bold")

ax.set_title("Policy economics — total financial cost and composition "
             "(model at t = 0.50)",
             fontsize=13.5, fontweight="bold", pad=16)
ax.set_ylabel("Total financial cost ($)", fontsize=11.5, fontweight="bold")
ax.set_ylim(0, y_max)
ax.yaxis.set_major_formatter("${x:,.0f}")
ax.grid(axis="y", ls="--", alpha=0.3)
ax.legend(loc="upper right", fontsize=9.5, frameon=True, edgecolor="#343a40")
plt.tight_layout()
plt.show()

#  Executive summary 
model = policies[2]
saved = null_cost - model["cost"]
b     = np.array(boot[model["policy"]])

print("=" * 72)
print("BUSINESS SUMMARY — cost-aware retention policy (t = 0.50)")
print("=" * 72)
print(f"Cost parameters    : C_FN = ${C_FN:,.0f}  |  C_FP = ${C_FP:,.0f}  "
      f"(ratio {C_FN/C_FP:.2f}:1)")
print(f"Break-even P(churn): {p_breakeven:.4f}")
print()
print(f"Model @ t = 0.50")
print(f"  Targeted         : {model['n_target']:,} "
      f"({model['n_target']/n_test*100:.1f}% of cohort)")
print(f"  Churners caught  : {model['tp']:,}  ({model['recall']*100:.1f}% recall)")
print(f"  Wasted offers    : {model['fp']:,}  × ${C_FP:,.0f}  = "
      f"${model['fp']*C_FP:,.0f}")
print(f"  Silent attrition : {model['fn']:,}  × ${C_FN:,.0f}  = "
      f"${model['fn']*C_FN:,.0f}")
print(f"  Total cost       : ${model['cost']:,.0f}")
print(f"  95% bootstrap CI : [${np.percentile(b, 2.5):,.0f}, "
      f"${np.percentile(b, 97.5):,.0f}]")
print()
print(f"Versus null action : saves ${saved:,.0f}  "
      f"({saved/null_cost*100:.1f}% reduction)")
print(f"Versus universal   : saves ${policies[1]['cost']-model['cost']:,.0f}")
print("=" * 72)

### 13.1 Empirical Out-of-Fold Threshold Optimization ($t^*$)
* Sweeps intervention thresholds from 0.05 to 0.50 across out-of-fold validation predictions to prevent threshold-tuning leakage.
* Pinpoints the operational threshold $t^*$ that minimizes total business cost across the portfolio.

In [ ]:
from sklearn.metrics import confusion_matrix

#  Economic parameters 
C_FN = 700.0     # lost CLV per missed churner
C_FP = 150.0     # retention-offer cost per targeted non-churner
p_breakeven = C_FP / (C_FN + C_FP)

#  Data 
y_test_arr = np.asarray(final_model["y_true"])
p_test     = np.asarray(final_model["y_pred_proba"])
n_test     = len(y_test_arr)
n_churners = int(y_test_arr.sum())

oof_proba = np.asarray(oof["LogisticRegression"])
y_oof     = np.asarray(y_hp)


def _cost(y_true, y_pred):
    _, fp, fn, _ = confusion_matrix(y_true, y_pred).ravel()
    return float(fn * C_FN + fp * C_FP)


#  Sweep thresholds on OOF
thresh_candidates = np.linspace(0.05, 0.50, 46)

sweep = pd.DataFrame([{
    "Threshold": round(float(t), 3),
    "Test_Cost": _cost(y_test_arr, (p_test >= t).astype(int)),
    "OOF_Cost":  _cost(y_oof,      (oof_proba >= t).astype(int)),
} for t in thresh_candidates])

t_opt = float(sweep.loc[sweep["OOF_Cost"].idxmin(), "Threshold"])
print(f"Cost-optimal threshold (OOF) : t* = {t_opt:.3f}")
print(f"Break-even probability       : {p_breakeven:.3f}")


#  Per-threshold breakdown on the TEST set (reporting) 
def _row(t):
    preds = (p_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_arr, preds).ravel()
    return {
        "Threshold":  t,
        "TP": int(tp), "FP": int(fp), "FN": int(fn),
        "Recall":     tp / (tp + fn) if (tp + fn) else 0.0,
        "Precision":  tp / (tp + fp) if (tp + fp) else 0.0,
        "Total Cost": float(fn * C_FN + fp * C_FP),
        "FN Cost":    float(fn * C_FN),
        "FP Cost":    float(fp * C_FP),
    }


df_sweep = pd.DataFrame([_row(t) for t in thresh_candidates])
best_row = df_sweep.loc[df_sweep["Total Cost"].idxmin()]

null_cost      = n_churners * C_FN
universal_cost = (n_test - n_churners) * C_FP


#  Summary table — four operating points 
default_row = df_sweep.loc[(df_sweep["Threshold"] - 0.50).abs().idxmin()]

summary = pd.DataFrame([
    {"Policy":         "1. Null action",
     "Threshold":      "—",
     "Targeted":       0,
     "Recall":         "0.0%",
     "Precision":      "—",
     "Cost":           null_cost,
     "Saved vs. null": 0.0},
    {"Policy":         "2. Universal concession",
     "Threshold":      "t = 0.00",
     "Targeted":       n_test,
     "Recall":         "100.0%",
     "Precision":      f"{n_churners/n_test*100:.1f}%",
     "Cost":           universal_cost,
     "Saved vs. null": null_cost - universal_cost},
    {"Policy":         "3. Model @ t = 0.50",
     "Threshold":      "t = 0.50",
     "Targeted":       int(default_row["TP"] + default_row["FP"]),
     "Recall":         f"{default_row['Recall']*100:.1f}%",
     "Precision":      f"{default_row['Precision']*100:.1f}%",
     "Cost":           default_row["Total Cost"],
     "Saved vs. null": null_cost - default_row["Total Cost"]},
    {"Policy":         f"4. Model @ t* = {t_opt:.2f}",
     "Threshold":      f"t = {t_opt:.2f}",
     "Targeted":       int(best_row["TP"] + best_row["FP"]),
     "Recall":         f"{best_row['Recall']*100:.1f}%",
     "Precision":      f"{best_row['Precision']*100:.1f}%",
     "Cost":           best_row["Total Cost"],
     "Saved vs. null": null_cost - best_row["Total Cost"]},
]).set_index("Policy")

summary["Cost"]           = summary["Cost"].map(lambda v: f"${v:,.0f}")
summary["Saved vs. null"] = summary["Saved vs. null"].map(lambda v: f"${v:,.0f}")
display(summary)


#  Cost-vs-threshold curve with markers 
fig, ax = plt.subplots(figsize=(11.5, 6), dpi=180)

# Main cost curves
ax.plot(df_sweep["Threshold"], df_sweep["Total Cost"],
        color=PALETTE[0], lw=3.0, zorder=3,
        label="Total cost (test set)")

ax.plot(df_sweep["Threshold"], df_sweep["FN Cost"],
        color="#e67e22", lw=2.2, zorder=2,
        label=f"Silent churn cost (FN × ${C_FN:,.0f})")

ax.plot(df_sweep["Threshold"], df_sweep["FP Cost"],
        color="#27ae60", lw=2.2, zorder=2,
        label=f"Wasted offers (FP × ${C_FP:,.0f})")

# Reference baselines
ax.axhline(null_cost, color="#7f8c8d", ls="--", lw=1.6, alpha=0.9,
           label=f"Null action (${null_cost:,.0f})")
ax.axhline(universal_cost, color="#8e44ad", ls="--", lw=1.6, alpha=0.9,
           label=f"Universal concession (${universal_cost:,.0f})")

#  Cross-section: where FN cost = FP cost 
cross_idx  = (df_sweep["FN Cost"] - df_sweep["FP Cost"]).abs().idxmin()
t_cross    = float(df_sweep.loc[cross_idx, "Threshold"])
cost_cross = float(df_sweep.loc[cross_idx, "FN Cost"])

ax.scatter([t_cross], [cost_cross], color="black", s=110, zorder=8,
           marker="o", edgecolor="white", linewidth=1.8,
           label=f"FN / FP cross-section (t = {t_cross:.2f})")
ax.annotate(
    f"Cross-section\n t = {t_cross:.2f}   |   ${cost_cross:,.0f}",
    xy=(t_cross, cost_cross),
    xytext=(t_cross + 0.05, cost_cross * 1.85),
    arrowprops=dict(facecolor="black", shrink=0.05, width=1.3, headwidth=6),
    fontsize=9.5, fontweight="bold", ha="left", va="center",
    bbox=dict(boxstyle="round,pad=0.4", fc="#f8f9fa", ec="black", lw=1.4),
)

#  Optimal threshold marker 
ax.scatter([t_opt], [best_row["Total Cost"]],
           color="#dc2626", s=150, zorder=9,
           marker="o", edgecolor="white", linewidth=2.0,
           label=f"Optimal t* = {t_opt:.2f}")
ax.annotate(
    f"Optimal t* = {t_opt:.2f}\n${best_row['Total Cost']:,.0f}",
    xy=(t_opt, best_row["Total Cost"]),
    xytext=(t_opt - 0.03, best_row["Total Cost"] * 1.42),
    arrowprops=dict(facecolor="#dc2626", shrink=0.05, width=1.3, headwidth=6),
    fontsize=9.5, fontweight="bold", ha="right", va="center",
    color="#a01a1a",
    bbox=dict(boxstyle="round,pad=0.4", fc="#fff5f5", ec="#dc2626", lw=1.6),
)

# Break-even vertical line
ax.axvline(p_breakeven, color="green", ls="-.", lw=1.3, alpha=0.8,
           label=f"Break-even p = {p_breakeven:.2f}")

# Formatting
ax.set_xlabel("Decision cutoff threshold  t",
              fontsize=11.5, fontweight="bold")
ax.set_ylabel("Financial cost on test set ($)",
              fontsize=11.5, fontweight="bold")
ax.set_title("Cost-optimal threshold search — tuned on OOF, evaluated on test",
             fontsize=12.5, fontweight="bold", pad=12)
ax.yaxis.set_major_formatter("${x:,.0f}")
ax.grid(ls=":", alpha=0.5)
ax.legend(loc="upper right", fontsize=9, framealpha=0.95)
plt.tight_layout()
plt.show()


#  Executive summary 
print("=" * 72)
print("OPTIMAL-COST THRESHOLD — SUMMARY")
print("=" * 72)
print(f"Cost parameters        : C_FN = ${C_FN:,.0f}  |  C_FP = ${C_FP:,.0f}  "
      f"(ratio {C_FN/C_FP:.2f}:1)")
print(f"Break-even P(churn)    : {p_breakeven:.4f}")
print(f"Optimal threshold (OOF): t* = {t_opt:.3f}")
print(f"FN/FP cross-section    : t  = {t_cross:.3f}")
print()
print(f"Model @ t = 0.50   →  ${default_row['Total Cost']:,.0f}")
print(f"Model @ t* = {t_opt:.2f}  →  ${best_row['Total Cost']:,.0f}")
print(f"Threshold tuning saves : "
      f"${default_row['Total Cost'] - best_row['Total Cost']:,.0f}")
print()
print(f"At t* = {t_opt:.2f}:")
print(f"  Targeted        : {int(best_row['TP'] + best_row['FP']):,} "
      f"({(best_row['TP'] + best_row['FP']) / n_test * 100:.1f}% of cohort)")
print(f"  Churners caught : {int(best_row['TP']):,} "
      f"({best_row['Recall']*100:.1f}% recall)")
print(f"  Missed churners : {int(best_row['FN']):,} "
      f"(× ${C_FN:,.0f} = ${best_row['FN Cost']:,.0f})")
print(f"  Wasted offers   : {int(best_row['FP']):,} "
      f"(× ${C_FP:,.0f} = ${best_row['FP Cost']:,.0f})")
print(f"  Total cost      : ${best_row['Total Cost']:,.0f}  "
      f"({(null_cost - best_row['Total Cost'])/null_cost*100:.1f}% below null)")
print("=" * 72)

# Persist tuned threshold for downstream cells
final_model["threshold"] = t_opt

---
## 14. Business Impact & Policy Comparison

### Comparing Operational Strategies on Holdout Data
1. **Null Policy (Status Quo / Inaction):** Zero retention spending. Every churner leaves unaddressed ($\text{FN} \times \$700$).
2. **Universal Concession (Blind Retention Campaign):** Send \$150 discounts to the entire customer base ($\text{FP} \times \$150$). Highly inefficient.
3. **Naive AI Policy ($t = 0.50$):** Standard statistical cutoff. Overly conservative; misses many accounts with a 25%–45% churn risk.
4. **Cost-Optimized AI Policy ($t = t^*$):** Mathematically optimal intervention point targeting accounts where expected loss exceeds intervention cost.

In [ ]:
# Final diagnostics — evaluated at the cost-optimal threshold t*
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve,
    roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score,
)
from sklearn.calibration import calibration_curve

#  Inputs
y_true = np.asarray(final_model["y_true"])
y_prob = np.asarray(final_model["y_pred_proba"])
name   = final_model["name"]
t_star = float(final_model["threshold"])      # cost-optimal threshold
t_def  = 0.50
prevalence = y_true.mean()

y_pred_star = (y_prob >= t_star).astype(int)
y_pred_def  = (y_prob >= t_def).astype(int)

print(f"Model           : {name}")
print(f"Chosen cutoff   : t* = {t_star:.3f}  (cost-optimal, tuned on OOF)")
print(f"Reference cutoff: t  = {t_def:.2f}   (default)")
print(f"Test prevalence : {prevalence:.4f}")



# 1. Confusion matrix — side by side at both thresholds

fig_cm, axes_cm = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

for ax, thr, y_pred, title in [
    (axes_cm[0], t_def,  y_pred_def,  f"Default threshold  t = {t_def:.2f}"),
    (axes_cm[1], t_star, y_pred_star, f"Cost-optimal threshold  t* = {t_star:.2f}"),
]:
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Retained (0)", "Churn (1)"]).plot(
        cmap="Blues", values_format="d", ax=ax, colorbar=False,
    )
    ax.set_title(title, fontweight="bold", fontsize=11.5, pad=10)
    ax.grid(False)

plt.suptitle(f"Confusion matrices — {name}",
             fontweight="bold", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()



# 2. Metric comparison — default vs cost-optimal

def _metrics(y_true, y_pred, y_prob, thr):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Threshold":  f"t = {thr:.2f}",
        "Targeted":   int(tp + fp),
        "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
        "Recall":     f"{recall_score(y_true, y_pred):.4f}",
        "Precision":  f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "F1":         f"{f1_score(y_true, y_pred, zero_division=0):.4f}",
        "Accuracy":   f"{accuracy_score(y_true, y_pred):.4f}",
    }

metric_df = pd.DataFrame([
    _metrics(y_true, y_pred_def,  y_prob, t_def),
    _metrics(y_true, y_pred_star, y_prob, t_star),
]).set_index("Threshold")

display(metric_df)

# Threshold-independent metrics printed once
auc_val    = roc_auc_score(y_true, y_prob)
pr_auc_val = average_precision_score(y_true, y_prob)
brier_val  = brier_score_loss(y_true, y_prob)
print(f"ROC-AUC  : {auc_val:.4f}   (threshold-independent)")
print(f"PR-AUC   : {pr_auc_val:.4f}   (threshold-independent)")
print(f"Brier    : {brier_val:.4f}   (lower is better; threshold-independent)")



# 3. ROC + PR + Calibration (3-panel)

fig_diag, axes = plt.subplots(1, 3, figsize=(17, 5), dpi=150)
primary = PALETTE[0]

#  Panel 1: ROC curve
ax = axes[0]
fpr, tpr, thr_roc = roc_curve(y_true, y_prob)
ax.plot(fpr, tpr, color=primary, lw=2.4,
        label=f"{name}  (ROC-AUC = {auc_val:.4f})")
ax.plot([0, 1], [0, 1], "k:", lw=1.2, label="Random chance")

# Mark the operating point for the chosen threshold
idx_roc = int(np.argmin(np.abs(thr_roc - t_star)))
ax.scatter(fpr[idx_roc], tpr[idx_roc],
           color="#dc2626", s=110, zorder=6,
           edgecolor="white", linewidth=1.6,
           label=f"Operating point (t* = {t_star:.2f})")
ax.annotate(
    f"t* = {t_star:.2f}\nTPR = {tpr[idx_roc]:.2f}\nFPR = {fpr[idx_roc]:.2f}",
    xy=(fpr[idx_roc], tpr[idx_roc]),
    xytext=(fpr[idx_roc] + 0.12, tpr[idx_roc] - 0.20),
    arrowprops=dict(facecolor="#dc2626", shrink=0.05, width=1.2, headwidth=6),
    fontsize=9, fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.35", fc="#fff5f5", ec="#dc2626", lw=1.4),
)
ax.set_title("ROC curve", fontweight="bold", fontsize=11.5)
ax.set_xlabel("False positive rate", fontweight="bold")
ax.set_ylabel("True positive rate", fontweight="bold")
ax.legend(loc="lower right", fontsize=9, framealpha=0.95)
ax.grid(ls=":", alpha=0.5)

#  Panel 2: Precision-Recall curve
ax = axes[1]
prec, rec, thr_pr = precision_recall_curve(y_true, y_prob)
ax.plot(rec, prec, color=primary, lw=2.4,
        label=f"{name}  (PR-AUC = {pr_auc_val:.4f})")
ax.axhline(prevalence, color="k", ls=":", lw=1.2,
           label=f"Base prevalence ({prevalence:.3f})")

# Operating point for the chosen threshold
idx_pr = int(np.argmin(np.abs(thr_pr - t_star)))
ax.scatter(rec[idx_pr], prec[idx_pr],
           color="#dc2626", s=110, zorder=6,
           edgecolor="white", linewidth=1.6,
           label=f"Operating point (t* = {t_star:.2f})")
ax.annotate(
    f"t* = {t_star:.2f}\nPrec = {prec[idx_pr]:.2f}\nRec  = {rec[idx_pr]:.2f}",
    xy=(rec[idx_pr], prec[idx_pr]),
    xytext=(rec[idx_pr] + 0.10, prec[idx_pr] + 0.10),
    arrowprops=dict(facecolor="#dc2626", shrink=0.05, width=1.2, headwidth=6),
    fontsize=9, fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.35", fc="#fff5f5", ec="#dc2626", lw=1.4),
)
ax.set_title("Precision-recall curve", fontweight="bold", fontsize=11.5)
ax.set_xlabel("Recall", fontweight="bold")
ax.set_ylabel("Precision", fontweight="bold")
ax.legend(loc="lower left", fontsize=9, framealpha=0.95)
ax.grid(ls=":", alpha=0.5)

#  Panel 3: Calibration / reliability diagram
ax = axes[2]
prob_true, prob_pred = calibration_curve(y_true, y_prob,
                                         n_bins=10, strategy="uniform")
ax.plot(prob_pred, prob_true, "s-", color=primary, lw=2.2,
        label=f"{name}  (Brier = {brier_val:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Perfect calibration")

# Mark where the chosen threshold sits on the reliability curve
ax.axvline(t_star, color="#dc2626", ls="-.", lw=1.4, alpha=0.8,
           label=f"Chosen cutoff  t* = {t_star:.2f}")
ax.axvline(t_def,  color="black",   ls=":",  lw=1.2, alpha=0.7,
           label=f"Default cutoff  t = {t_def:.2f}")

ax.set_title("Calibration diagram", fontweight="bold", fontsize=11.5)
ax.set_xlabel("Mean predicted probability (10 bins)", fontweight="bold")
ax.set_ylabel("Empirical fraction of positives", fontweight="bold")
ax.legend(loc="upper left", fontsize=8.5, framealpha=0.95)
ax.grid(ls=":", alpha=0.5)

plt.suptitle(f"Diagnostic dashboard — {name}",
             fontweight="bold", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()



# 4. Executive summary

tn_s, fp_s, fn_s, tp_s = confusion_matrix(y_true, y_pred_star).ravel()
tn_d, fp_d, fn_d, tp_d = confusion_matrix(y_true, y_pred_def).ravel()

print("=" * 74)
print(f"FINAL OPERATING POINT — {name} @ t* = {t_star:.3f}")
print("=" * 74)
print(f"  Test rows        : {len(y_true):,}")
print(f"  Test churners    : {int(y_true.sum()):,}  (prevalence {prevalence:.2%})")
print()
print(f"  Targeted         : {int(tp_s + fp_s):,}  "
      f"({(tp_s + fp_s) / len(y_true) * 100:.1f}% of cohort)")
print(f"  True positives   : {int(tp_s):,}")
print(f"  False negatives  : {int(fn_s):,}   (silent churners)")
print(f"  False positives  : {int(fp_s):,}   (wasted offers)")
print()
print(f"  Recall           : {tp_s / (tp_s + fn_s):.4f}")
print(f"  Precision        : {tp_s / (tp_s + fp_s):.4f}")
print(f"  F1               : {2 * tp_s / (2 * tp_s + fp_s + fn_s):.4f}")
print()
print(f"  ROC-AUC          : {auc_val:.4f}")
print(f"  PR-AUC           : {pr_auc_val:.4f}")
print(f"  Brier            : {brier_val:.4f}")
print()
print(f"Delta vs. default t = 0.50:")
print(f"  Recall  : {tp_s / (tp_s + fn_s) - tp_d / (tp_d + fn_d):+.4f}  "
      f"({tp_s / (tp_s + fn_s) * 100:.1f}% vs {tp_d / (tp_d + fn_d) * 100:.1f}%)")
print(f"  Precision: {tp_s / (tp_s + fp_s) - tp_d / (tp_d + fp_d):+.4f}  "
      f"({tp_s / (tp_s + fp_s) * 100:.1f}% vs {tp_d / (tp_d + fp_d) * 100:.1f}%)")
print(f"  F1       : "
      f"{2 * tp_s / (2 * tp_s + fp_s + fn_s) - 2 * tp_d / (2 * tp_d + fp_d + fn_d):+.4f}")
print("=" * 74)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# Inputs from the threshold-search cell 
y_true = np.asarray(final_model["y_true"])
p_test = np.asarray(final_model["y_pred_proba"])
# Safely get t_star (using t_opt if the old key was removed)
t_star = float(final_model.get("threshold", getattr(pd, 't_opt', 0.17))) 
t_def  = 0.50                                     

# Pull from the business-insight cell (must have run first)
# null_cost, universal_cost, n_churners, n_test, C_FN, C_FP are already defined.

def _cm_at(thr):
    """Confusion matrix at a given threshold."""
    return confusion_matrix(y_true, (p_test >= thr).astype(int))

cm_50  = _cm_at(t_def)
cm_opt = _cm_at(t_star)

_, fp_50,  fn_50,  tp_50  = cm_50.ravel()
_, fp_opt, fn_opt, tp_opt = cm_opt.ravel()

TARGET_NAMES = ["No", "Yes"]

print("=" * 74)
print(f"DEFAULT THRESHOLD  t = {t_def:.2f}")
print("=" * 74)
print(classification_report(y_true, (p_test >= t_def).astype(int), target_names=TARGET_NAMES, digits=2, zero_division=0))

print("\n" + "=" * 74)
print(f"COST-OPTIMAL THRESHOLD  t* = {t_star:.2f}")
print("=" * 74)
print(classification_report(y_true, (p_test >= t_star).astype(int), target_names=TARGET_NAMES, digits=2, zero_division=0))

# Charging $150 to everyone, 100% Save Rate Baseline
ASSUMED_SAVE_RATE = 1.00

policy_labels = [
    "1. Null action\n(zero intervention)",
    "2. Universal\n(blanket outreach)",
    f"3. Model\n(t = {t_def:.2f}, default)",
    f"4. Model\n(t* = {t_star:.2f}, cost-opt)",
]

# 1. Unsaved Churn Loss: (Total Churners - (Saved Churners)) * $700
# Only a percentage of True Positives are actually saved.
loss_costs = [
    float((n_churners - (0 * ASSUMED_SAVE_RATE)) * C_FN),           # null
    float((n_churners - (n_churners * ASSUMED_SAVE_RATE)) * C_FN),  # universal
    float((n_churners - (tp_50 * ASSUMED_SAVE_RATE)) * C_FN),       # model @ 0.50
    float((n_churners - (tp_opt * ASSUMED_SAVE_RATE)) * C_FN),      # model @ t*
]

# 2. Campaign Spend: All Targeted Customers * $150
spend_costs = [
    0.0,                                       # null: no offers
    float(n_test * C_FP),                      # universal: offers to everyone
    float((tp_50 + fp_50) * C_FP),             # model @ 0.50
    float((tp_opt + fp_opt) * C_FP),           # model @ t*
]

totals = [loss + spend for loss, spend in zip(loss_costs, spend_costs)]
null_total = totals[0]
saved_vals = [null_total - t for t in totals]  # net capital saved vs. null

fig_biz, axes_b = plt.subplots(1, 2, figsize=(16, 6.5), dpi=180)

COLOR_FN = "#c0392b"
COLOR_FP = "#f39c12"

# Panel 1: Cost composition (stacked bars) 
ax = axes_b[0]
ax.bar(policy_labels, loss_costs, width=0.55,
       color=COLOR_FN, edgecolor="black", linewidth=1.2, alpha=0.9,
       label=f"Unsaved Churn Loss  (Missed × ${C_FN:,.0f})")
ax.bar(policy_labels, spend_costs, bottom=loss_costs, width=0.55,
       color=COLOR_FP, edgecolor="black", linewidth=1.2, alpha=0.9,
       label=f"Total Campaign Spend  (Targeted × ${C_FP:,.0f})")

y_max1 = max(totals) * 1.22
for i, (loss, spend, tot) in enumerate(zip(loss_costs, spend_costs, totals)):
    ax.text(i, tot + y_max1 * 0.02, f"${tot:,.0f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

    # Loss segment (Bottom)
    if loss > tot * 0.12:
        ax.text(i, loss / 2, f"Loss\n${loss/1000:.0f}k\n({loss/tot:.0%})",
                ha="center", va="center", color="white",
                fontsize=9, fontweight="bold")
    elif loss > 0:
        ax.annotate(f"Loss ${loss/1000:.0f}k ({loss/tot:.0%})",
                    xy=(i + 0.27, loss / 2), xytext=(i + 0.40, loss / 2),
                    ha="left", va="center", fontsize=8.5, fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color="black", lw=1.1))

    # Spend segment (Top)
    if spend > tot * 0.12:
        ax.text(i, loss + spend / 2, f"Spend\n${spend/1000:.0f}k\n({spend/tot:.0%})",
                ha="center", va="center", color="white",
                fontsize=9, fontweight="bold")
    elif spend > 0:
        ax.annotate(f"Spend ${spend/1000:.0f}k ({spend/tot:.0%})",
                    xy=(i + 0.27, loss + spend / 2),
                    xytext=(i + 0.40, loss + spend / 2),
                    ha="left", va="center", fontsize=8.5, fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color="black", lw=1.1))

ax.set_title(f"Cost composition across operational policies (@ {ASSUMED_SAVE_RATE*100:.0f}% Save Rate)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel("Total financial cost ($)", fontsize=11, fontweight="bold")
ax.set_ylim(0, y_max1)
ax.yaxis.set_major_formatter("${x:,.0f}")
ax.tick_params(axis="x", labelsize=9.5)
ax.legend(loc="upper right", fontsize=9, frameon=True, edgecolor="#343a40")
ax.grid(axis="y", ls="--", alpha=0.3)

# Panel 2: Net capital saved vs. null 
ax = axes_b[1]
colors = ["#95a5a6", "#e67e22", "#3498db", "#27ae60"]
bars = ax.bar(policy_labels, saved_vals, width=0.55,
              color=colors, edgecolor="black", linewidth=1.2, alpha=0.9)

y_max2 = max(max(saved_vals) * 1.25, 5000)
y_min = min(min(saved_vals) * 1.15, -5000)

for bar, val in zip(bars, saved_vals):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2,
                val + (y_max2 - y_min) * 0.02,
                f"+${val:,.0f}",
                ha="center", va="bottom", fontsize=10, fontweight="bold")
    elif val < 0:
        ax.text(bar.get_x() + bar.get_width() / 2,
                val - (y_max2 - y_min) * 0.02,
                f"-${abs(val):,.0f}",
                ha="center", va="top", fontsize=10, fontweight="bold", color="#c0392b")
    else:
        ax.text(bar.get_x() + bar.get_width() / 2,
                (y_max2 - y_min) * 0.02,
                "$0\n(reference)",
                ha="center", va="bottom", fontsize=9.5,
                fontweight="bold", color="#7f8c8d")

ax.axhline(0, color="black", lw=1.2, alpha=0.7)
ax.set_title(f"Net capital protected vs. null action baseline (@ {ASSUMED_SAVE_RATE*100:.0f}% Save Rate)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel("Net capital saved ($)", fontsize=11, fontweight="bold")
ax.set_ylim(y_min, y_max2)
ax.yaxis.set_major_formatter("${x:,.0f}")
ax.tick_params(axis="x", labelsize=9.5)
ax.grid(axis="y", ls="--", alpha=0.3)

plt.tight_layout()
plt.show()

# Executive summary
print("=" * 80)
print(f"COST COMPOSITION AND CAPITAL PROTECTION — SUMMARY (@ {ASSUMED_SAVE_RATE*100:.0f}% Save Rate)")
print("=" * 80)
print(f"{'Policy':<38} {'Total Cost':>14} {'Saved vs. null':>20}")
print("-" * 80)
for label, cost, saved in zip(policy_labels, totals, saved_vals):
    clean = label.replace("\n", " ")
    saved_str = f"${saved:,.0f}" + (" (ref)" if saved == 0 else "")
    print(f"  {clean:<36} ${cost:>12,.0f} {saved_str:>20}")
print("-" * 80)

incremental = totals[2] - totals[3]
print(f"Threshold tuning (t = 0.50 → t* = {t_star:.2f}): "
      f"${incremental:,.0f} additional profit")
print(f"Total model vs. universal: "
      f"${totals[1] - totals[3]:,.0f}")
print("=" * 80)

### 14.1 Sensitivity Analysis

A common critique of deterministic retention economics is the **"Perfect Save Assumption"**—the premise that extending a concession prevents 100% of impending churn events. In real commercial operations:
1. **Save Rate Elasticity ($\gamma$):** Only a fraction of churners accept the counter-offer and renew ($\gamma \in [0.20, 0.80]$). The remaining $1 - \gamma$ take the concession and leave anyway (or churn despite outreach).
2. **Concession vs. CLV Volatility:** The financial ratio $C_{FP} / C_{FN}$ varies by customer tier, product margin, and promotional budget.

#### Adjusted Cost Formulation
Accounting for an imperfect save rate $\gamma \in (0, 1]$:
* **Missed Churner (FN):** Incurs full unmitigated CLV loss $\rightarrow C_{FN} = \$700$
* **Wasted Concession (FP):** Concession spent on a loyal customer $\rightarrow C_{FP} = \$150$
* **Contacted Churner (TP):** Receives concession $C_{FP}$; with probability $\gamma$ CLV is preserved, while with probability $1 - \gamma$ the customer churns regardless:
$$\text{Cost}(TP) = C_{FP} + (1 - \gamma) C_{FN}$$

The generalized portfolio cost function is:
$$\text{Total Cost}(t; \gamma, C_{FP}, C_{FN}) = \text{FN}(t) \cdot C_{FN} + \text{FP}(t) \cdot C_{FP} + \text{TP}(t) \cdot \left[ C_{FP} + (1 - \gamma) C_{FN} \right]$$

#### Analytical Break-Even Condition
An account with churn probability $p$ justifies an intervention only when the expected cost of action is less than inaction:
$$C_{FP} + (1 - \gamma) p C_{FN} \le p C_{FN} \iff p \ge \frac{C_{FP}}{\gamma \cdot C_{FN}}$$

In [ ]:
y_test_arr = np.asarray(final_model["y_true"])
p_test = np.asarray(final_model["y_pred_proba"])
t_star = float(final_model.get("threshold", 0.17))
t_def = 0.50

OFFER_COST = 150.0
CLV = 700.0 

def net_benefit_vs_null(y_true, y_prob, threshold, save_rate):
    """
    Corrected framework: 
    Net Benefit = (True Positives * save_rate * CLV) - (All Targeted * OFFER_COST)
    """
    targeted = (y_prob >= threshold).astype(int)
    true_positives = ((targeted == 1) & (y_true == 1)).sum()
    total_targeted = targeted.sum()
    
    return (true_positives * save_rate * CLV) - (total_targeted * OFFER_COST)

# 2. Sweep over the save rates the reviewer requested
save_rates = [1.0, 0.70, 0.50, 0.30]

row_opt = {"Policy": f"Model @ t* = {t_star:.2f}"}
row_50 = {"Policy": "Model @ t = 0.50"}
row_univ = {"Policy": "Universal offer"}
row_breakeven = {"Policy": "Break-even prob"}

plot_results = {}

# Calculate outcomes for each policy across the different save rates
for sr in save_rates:
    col_name = f"{sr*100:.0f}%"
    
    b_opt = net_benefit_vs_null(y_test_arr, p_test, t_star, sr)
    b_50 = net_benefit_vs_null(y_test_arr, p_test, t_def, sr)
    b_univ = net_benefit_vs_null(y_test_arr, p_test, 0.0, sr)
    be_prob = OFFER_COST / (CLV * sr)
    
    row_opt[col_name] = b_opt
    row_50[col_name] = b_50
    row_univ[col_name] = b_univ
    row_breakeven[col_name] = be_prob
    
    # Store for bar charts (Null is always 0)
    plot_results[sr] = [0.0, b_univ, b_50, b_opt]

# --- 3. Visualization: 2x2 Grid of Net Benefit Bar Charts ---
fig, axes = plt.subplots(2, 2, figsize=(16, 11), dpi=150)
axes = axes.flatten()

policy_labels = [
    "1. Null action\n(zero intervention)",
    "2. Universal\n(blanket outreach)",
    f"3. Model\n(t = {t_def:.2f})",
    f"4. Model\n(t* = {t_star:.2f})"
]
colors = ["#95a5a6", "#e67e22", "#3498db", "#27ae60"]

for i, sr in enumerate(save_rates):
    ax = axes[i]
    vals = plot_results[sr]
    
    bars = ax.bar(policy_labels, vals, width=0.55,
                  color=colors, edgecolor="black", linewidth=1.2, alpha=0.9)
    
    # Dynamic axis scaling per chart for readability
    y_max = max(max(vals) * 1.25, 5000)
    y_min = min(min(vals) * 1.15, -5000)
    
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, val + (y_max - y_min) * 0.02,
                    f"+${val:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
        elif val < 0:
            ax.text(bar.get_x() + bar.get_width() / 2, val - (y_max - y_min) * 0.02,
                    f"-${abs(val):,.0f}", ha="center", va="top", fontsize=10, fontweight="bold", color="#c0392b")
        else:
            ax.text(bar.get_x() + bar.get_width() / 2, (y_max - y_min) * 0.02,
                    "$0\n(reference)", ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#7f8c8d")

    ax.axhline(0, color="black", lw=1.2, alpha=0.7)
    ax.set_title(f"Net Capital Saved vs Null (@ {sr*100:.0f}% Save Rate)", fontsize=12, fontweight="bold", pad=12)
    ax.set_ylim(y_min, y_max)
    ax.yaxis.set_major_formatter("${x:,.0f}")
    ax.tick_params(axis="x", labelsize=9.5)
    ax.grid(axis="y", ls="--", alpha=0.3)
    
    # Only add y-labels to the left-most charts
    if i % 2 == 0:
        ax.set_ylabel("Net Expected Benefit ($)", fontsize=11, fontweight="bold")

plt.tight_layout(pad=2.0)
plt.show()

# --- 4. Formatted Reporting Table ---
df_sens = pd.DataFrame([row_opt, row_50, row_univ, row_breakeven])

def format_cell(val):
    if isinstance(val, float) and val <= 1.0: # format probabilities
        return f"{val:.2f}"
    if val > 0:
        return f"+${val:,.0f}"
    elif val < 0:
        return f"-${abs(val):,.0f}"
    return "$0"

for col in [f"{sr*100:.0f}%" for sr in save_rates]:
    df_sens[col] = df_sens[col].apply(format_cell)
display(df_sens.set_index("Policy"))

### 14.2 Individualized Expected Value (EV) Targeting

#### The Flaw in Flat Valuation
The portfolio-wide threshold ($t^* \approx 0.17$) derived in Section 13 assumes an identical loss for every churner ($C_{FN} = \$700$). In real commercial operations, subscription accounts exhibit substantial revenue heterogeneity:
* **Low-Tier Accounts:** Paying $\approx \$20/\text{month}$ ($\text{Annual CLV} \approx \$240$). Extending a $\$150$ concession burns capital even if their churn probability is high ($p = 0.35$), because expected recovered revenue ($0.35 \times 0.50 \times \$240 = \$42$) falls far short of the $\$150$ outlay.
* **Premium Accounts:** Paying $\approx \$110/\text{month}$ ($\text{Annual CLV} \approx \$1,320$). Ignoring an account at $p = 0.16$ simply because it sits just below a static $0.17$ cutoff forfeits an opportunity to protect substantial recurring revenue.

To eliminate this misallocation, we transition from an aggregate probability cutoff to **individualized Expected Value (EV) capital allocation**.

---

#### Mathematical Formulation
For each individual customer $i$, expected retention value $\mathbb{E}[V_i]$ is defined as:

$$\mathbb{E}[V_i] = \underbrace{\left( p_i \times \gamma \times \text{CLV}_i \right)}_{\text{Expected Capital Preserved}} - \underbrace{C_{FP}}_{\text{Offer Cost}}$$

Where:
* $p_i \in [0, 1]$: Calibrated predicted churn risk from the model (`final_model["y_pred_proba"]`).
* $\gamma$: Conservative campaign save rate ($\gamma = 0.50$, meaning $50\%$ of contacted churners renew).
* $\text{CLV}_i$: Annual account value derived directly from billing records ($\text{MonthlyCharges}_i \times 12\text{ months}$).
* $C_{FP}$: Fixed retention concession expense ($\$150$).

#### Individualized Decision Rule
An intervention is launched on customer $i$ if and only if expected return is positive:

$$\mathbb{E}[V_i] > 0 \iff p_i > \frac{C_{FP}}{\gamma \times \text{CLV}_i} = \frac{150}{0.50 \times (12 \times \text{MonthlyCharges}_i)} = \frac{25}{\text{MonthlyCharges}_i}$$

* A **$\$25/\text{month}$ account** requires $p_i > 100\%$ $\rightarrow$ **Never targeted** (cannot justify a $\$150$ offer under a $50\%$ save rate).
* A **$\$100/\text{month}$ account** requires $p_i > 25\%$ $\rightarrow$ **Targeted aggressively** to insulate high-margin revenue.

The code below executes this dynamic policy on the blind test partition and compares realized P&L against the global threshold policy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Retrieve predictions, actuals, and individualized financials
y_test_arr = np.asarray(final_model["y_true"])
p_test = np.asarray(final_model["y_pred_proba"])

# Extract MonthlyCharges exactly aligned with the test set
monthly_charges = df_test["MonthlyCharges"].values

ASSUMED_REMAINING_MONTHS = 12
OFFER_COST = 150.0  
ASSUMED_SAVE_RATE = 0.50

# Individualized Customer Lifetime Value (CLV)
clv_array = monthly_charges * ASSUMED_REMAINING_MONTHS

# 2. Calculate Expected Value (EV) per customer
# EV = (Probability of Churn * Save Rate * Individual CLV) - Offer Cost
expected_value_per_customer = (p_test * ASSUMED_SAVE_RATE * clv_array) - OFFER_COST

# 3. EV-Based Targeting Rule: Target ONLY if the individual EV is positive
ev_targeted = (expected_value_per_customer > 0).astype(int)

# Calculate actual realized benefit on the test set using this personalized rule
ev_tp = (ev_targeted == 1) & (y_test_arr == 1)
ev_gross_saved = clv_array[ev_tp].sum() * ASSUMED_SAVE_RATE
ev_campaign_cost = ev_targeted.sum() * OFFER_COST
ev_net_benefit = ev_gross_saved - ev_campaign_cost

# Compare to the old "Global t* = 0.17" policy using individualized CLVs
t_star_old = 0.17
global_targeted = (p_test >= t_star_old).astype(int)
global_tp = (global_targeted == 1) & (y_test_arr == 1)
global_gross_saved = clv_array[global_tp].sum() * ASSUMED_SAVE_RATE
global_campaign_cost = global_targeted.sum() * OFFER_COST
global_net_benefit = global_gross_saved - global_campaign_cost

# --- 4. Visualization ---
policy_names = [
    "1. Null Action",
    f"2. Global Threshold\n(t = {t_star_old:.2f})",
    "3. Individualized EV\n(Target if EV > $0)"
]
benefits = [0.0, global_net_benefit, ev_net_benefit]
colors = ['#95a5a6', '#3498db', '#9b59b6']

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)
bars = ax.bar(policy_names, benefits, color=colors, edgecolor="black", width=0.55)

for bar, val in zip(bars, benefits):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, val + max(benefits) * 0.02,
                f"+${val:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    else:
        ax.text(bar.get_x() + bar.get_width() / 2, max(benefits) * 0.02,
                "$0", ha="center", va="bottom", fontsize=10, fontweight="bold", color="#7f8c8d")

ax.set_title(f"Net Profit: Global Probability vs. Individualized EV Targeting\n(@ {ASSUMED_SAVE_RATE*100:.0f}% Save Rate)", fontweight='bold', pad=12)
ax.set_ylabel("Realized Net Benefit ($)", fontweight='bold')
ax.axhline(0, color='black', lw=1.2)
ax.yaxis.set_major_formatter("${x:,.0f}")
plt.tight_layout()
plt.show()

# --- 5. Executive Summary ---
print("=" * 80)
print("INDIVIDUALIZED TARGETING — SUMMARY")
print("=" * 80)
print(f"CLV Assumption           : MonthlyCharges x {ASSUMED_REMAINING_MONTHS} months")
print(f"Concession Offer Cost    : ${OFFER_COST:,.0f} flat")
print(f"Total Test Portfolio     : {len(y_test_arr):,} customers")
print("-" * 80)
print(f"Targeted via Global t*   : {global_targeted.sum():,} customers targeted")
print(f"Global t* Net Profit     : ${global_net_benefit:,.0f}")
print("-" * 80)
print(f"Targeted via EV Rule     : {ev_targeted.sum():,} customers targeted")
print(f"EV Rule Net Profit       : ${ev_net_benefit:,.0f}")
print("-" * 80)
print(f"Lift from Personalization: +${ev_net_benefit - global_net_benefit:,.0f} additional profit vs static threshold")
print("=" * 80)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5.5), dpi=150)

# Realized Net Profit (P&L Comparison)
policies = [
    "1. Null\nAction",
    f"2. Global Cutoff\n(t = {t_star_old:.2f})",
    "3. Individual EV\n(Target if EV > $0)",
]
profits = [0.0, global_net_benefit, ev_net_benefit]
bar_colors = ["#95a5a6", "#2980b9", "#8e44ad"]

bars1 = ax1.bar(
    policies,
    profits,
    color=bar_colors,
    width=0.52,
    edgecolor="black",
    linewidth=0.8,
)

for b, val in zip(bars1, profits):
    h = b.get_height()
    if val > 0:
        ax1.text(
            b.get_x() + b.get_width() / 2,
            h + 1500,
            f"+${val:,.0f}",
            ha="center",
            va="bottom",
            fontsize=9.5,
            fontweight="bold",
        )
    else:
        ax1.text(
            b.get_x() + b.get_width() / 2,
            1500,
            "$0",
            ha="center",
            va="bottom",
            fontsize=9.5,
            fontweight="bold",
            color="#7f8c8d",
        )

ax1.set_title(
    "Realized Net Profit Comparison", fontweight="bold", fontsize=11, pad=12
)
ax1.set_ylabel("Net Capital Preserved ($)", fontweight="bold")
ax1.yaxis.set_major_formatter("${x:,.0f}")
ax1.set_ylim(-5000, max(profits) * 1.25)
ax1.grid(axis="y", ls=":", alpha=0.5)

# Capital Efficiency (Gross Saved vs. Concession Spend)
x_pos = np.arange(2)
w = 0.35

spend_vals = [global_campaign_cost, ev_campaign_cost]
gross_vals = [global_gross_saved, ev_gross_saved]

ax2.bar(
    x_pos - w / 2,
    gross_vals,
    w,
    label="Gross Revenue Preserved",
    color="#27ae60",
    edgecolor="black",
    lw=0.8,
)
ax2.bar(
    x_pos + w / 2,
    spend_vals,
    w,
    label="Concession Budget Spent",
    color="#e74c3c",
    edgecolor="black",
    lw=0.8,
)

for i in range(2):
    ax2.text(
        x_pos[i] - w / 2,
        gross_vals[i] + 3000,
        f"${gross_vals[i] / 1000:,.1f}k",
        ha="center",
        va="bottom",
        fontsize=8.5,
        fontweight="bold",
    )
    ax2.text(
        x_pos[i] + w / 2,
        spend_vals[i] + 3000,
        f"${spend_vals[i] / 1000:,.1f}k",
        ha="center",
        va="bottom",
        fontsize=8.5,
        fontweight="bold",
    )

ax2.set_xticks(x_pos)
ax2.set_xticklabels(
    [
        f"Global t* = {t_star_old:.2f}\n({global_targeted.sum():,} Targeted)",
        f"Dynamic EV\n({ev_targeted.sum():,} Targeted)",
    ],
    fontweight="bold",
)
ax2.set_title(
    "Spend vs. Revenue Preserved", fontweight="bold", fontsize=11, pad=12
)
ax2.set_ylabel("Capital Volume ($)", fontweight="bold")
ax2.yaxis.set_major_formatter("${x:,.0f}")
ax2.set_ylim(0, max(gross_vals) * 1.18)
ax2.grid(axis="y", ls=":", alpha=0.5)
ax2.legend(loc="upper right", fontsize=8.5, frameon=True)

# Individual Decision Frontier & Account Allocations

# Categorize cohort accounts relative to EV vs Static rules
only_global = (global_targeted == 1) & (ev_targeted == 0)
both_targeted = (global_targeted == 1) & (ev_targeted == 1)
only_ev = (global_targeted == 0) & (ev_targeted == 1)

# Plot customer scatter (subsampled slightly for visual crispness)
ax3.scatter(
    monthly_charges[both_targeted],
    p_test[both_targeted],
    color="#8e44ad",
    alpha=0.45,
    s=18,
    label=f"Targeted by Both ({both_targeted.sum():,})",
)
ax3.scatter(
    monthly_charges[only_global],
    p_test[only_global],
    color="#e67e22",
    alpha=0.55,
    s=20,
    label=f"Discarded Low-Tier ({only_global.sum():,})",
)
if only_ev.sum() > 0:
    ax3.scatter(
        monthly_charges[only_ev],
        p_test[only_ev],
        color="#27ae60",
        alpha=0.8,
        s=35,
        edgecolor="black",
        lw=0.5,
        label=f"Rescued Premium ({only_ev.sum():,})",
    )

# Theoretical EV Break-even curve: p* = 25 / MonthlyCharges
mc_curve = np.linspace(20, 120, 200)
p_curve = 25.0 / mc_curve

ax3.plot(
    mc_curve,
    p_curve,
    color="#2c3e50",
    lw=2.5,
    ls="-",
    label="EV Boundary: $p^* = \\frac{25}{\\text{MonthlyCharges}}$",
)
ax3.axhline(
    t_star_old,
    color="#2980b9",
    lw=1.8,
    ls="--",
    label=f"Static Cutoff $t^* = {t_star_old:.2f}$",
)

ax3.set_title(
    "Customer Allocation Decision Frontier",
    fontweight="bold",
    fontsize=11,
    pad=12,
)
ax3.set_xlabel("Monthly Charges ($)", fontweight="bold")
ax3.set_ylabel("Predicted Churn Risk (p)", fontweight="bold")
ax3.set_xlim(15, 122)
ax3.set_ylim(0.0, 1.0)
ax3.grid(ls=":", alpha=0.5)
ax3.legend(loc="upper right", fontsize=7.8, frameon=True)

plt.tight_layout()
plt.show()

---
## 15. Portfolio Scoring & Generalization Summary

### Final Verification
Scores the entire cohort (in-sample training vs. out-of-sample holdout test partitions) to confirm:
* Out-of-sample ROC-AUC and PR-AUC closely mirror cross-validation estimates, confirming the absence of overfitting.
* Brier calibration scores remain stable across operational environments.
* The customer file is exported with predicted churn probabilities and action recommendations for retention marketing teams.

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Full Portfolio Feature Extraction & Inference
feature_cols = NUMERIC_COLS + CATEGORICAL_COLS + BOOLEAN_COLS
df_all = df[feature_cols].reset_index(drop=True)
y_all = df[TARGET].astype(int).reset_index(drop=True)

if final_model["name"] == "LogisticRegression":
  X_all = build_linear_frame(df_all)
else:
  X_all = build_tree_frame(df_all)

# Score entire customer base (7,043 accounts)
p_all = final_clf.predict_proba(X_all)[:, 1]

t_star = float(final_model.get("threshold", 0.17))
t_def = 0.50
C_FN_VAL = float(globals().get("C_FN", 700.0))
C_FP_VAL = float(globals().get("C_FP", 150.0))

# 2. Generalization Audit: Train (In-Sample) vs. Test (Out-of-Sample)
train_mask = np.isin(df.index, df_train.index)
test_mask = ~train_mask

slices = [
    ("Train rows (in-sample)", train_mask),
    ("Test rows (out-of-sample)", test_mask),
    ("All rows (mixed portfolio)", np.ones(len(y_all), dtype=bool)),
]

slice_records = []
for name, mask in slices:
  y_sub = y_all.values[mask]
  p_sub = p_all[mask]
  slice_records.append({
      "Dataset Partition": name,
      "Accounts": int(mask.sum()),
      "Churn Rate": f"{y_sub.mean():.2%}",
      "ROC-AUC": round(roc_auc_score(y_sub, p_sub), 4),
      "PR-AUC": round(average_precision_score(y_sub, p_sub), 4),
      "Brier Score": round(brier_score_loss(y_sub, p_sub), 4),
  })

slice_df = pd.DataFrame(slice_records).set_index("Dataset Partition")
display(slice_df)

train_auc = slice_df.loc["Train rows (in-sample)", "ROC-AUC"]
test_auc = slice_df.loc["Test rows (out-of-sample)", "ROC-AUC"]
auc_gap = train_auc - test_auc

# 3. Macro Classification Audits across Decision Cutoffs
TARGET_NAMES = ["Retain (0)", "Churn (1)"]
for tag, thr in [
    ("DEFAULT THRESHOLD", t_def),
    ("COST-OPTIMAL THRESHOLD", t_star),
]:
  print("=" * 78)
  print(f"{tag}: Cutoff t = {thr:.2f} (Evaluated on all {len(y_all):,} rows)")
  print("=" * 78)
  print(
      classification_report(
          y_all,
          (p_all >= thr).astype(int),
          target_names=TARGET_NAMES,
          digits=3,
          zero_division=0,
      )
  )

# 4. Comprehensive Cost & Multi-Scenario Save Rate Benchmark
n_all = len(y_all)
n_churn_all = int(y_all.sum())
null_cost_all = n_churn_all * C_FN_VAL

SAVE_SCENARIOS = [0.30, 0.50, 0.70, 1.00]


def evaluate_policy_scenarios(thr_or_mode, y_vec, p_vec):
  """Computes metrics, percentages, and net dollars saved across save rate scenarios."""
  if thr_or_mode == "null":
    preds = np.zeros(len(y_vec), dtype=int)
  elif thr_or_mode == "universal":
    preds = np.ones(len(y_vec), dtype=int)
  else:
    preds = (p_vec >= thr_or_mode).astype(int)

  tn, fp, fn, tp = confusion_matrix(y_vec, preds).ravel()
  n_targeted = tp + fp
  pct_targeted = (n_targeted / n_all) * 100
  pct_churners_saved = (tp / n_churn_all) * 100 if n_churn_all > 0 else 0.0
  precision_saved = (tp / n_targeted) * 100 if n_targeted > 0 else 0.0
  concession_spend = n_targeted * C_FP_VAL

  base_total_cost = float(fn * C_FN_VAL + fp * C_FP_VAL)
  base_net_saved = null_cost_all - base_total_cost

  row = {
      "Targeted": f"{n_targeted:,} ({pct_targeted:.1f}%)",
      "Churners Saved (Recall)": f"{tp:,} ({pct_churners_saved:.1f}%)",
      "Save Precision (Hit Rate)": f"{precision_saved:.1f}%",
      "Total Cost": f"${base_total_cost:,.0f}",
      "Net Saved vs. Null": f"${base_net_saved:,.0f}",
      "_raw_cost": base_total_cost,
      "_raw_saved": base_net_saved,
  }

  # Multi-scenario net capital preserved
  for gamma in SAVE_SCENARIOS:
    gross_saved = tp * gamma * C_FN_VAL
    net_saved = gross_saved - concession_spend
    g_key = int(gamma * 100)
    row[f"Net Saved @ {g_key}% Save"] = (
        f"${net_saved:,.0f}" if net_saved >= 0 else f"-${abs(net_saved):,.0f}"
    )
    row[f"_raw_saved_{g_key}"] = net_saved

  return row


policies = [
    ("1. Null Action", "—", "null"),
    ("2. Universal Concession", "t = 0.00", "universal"),
    (f"3. Model @ t = {t_def:.2f}", f"t = {t_def:.2f}", t_def),
    (f"4. Model @ t* = {t_star:.2f}", f"t = {t_star:.2f}", t_star),
]

policy_rows = []
for name, thr_label, mode in policies:
  metrics = evaluate_policy_scenarios(mode, y_all.values, p_all)
  metrics["Operational Policy"] = name
  metrics["Threshold"] = thr_label
  policy_rows.append(metrics)

cost_summary_df = pd.DataFrame(policy_rows).set_index("Operational Policy")

print("=" * 90)
print(
    f"CAMPAIGN EFFICACY BENCHMARK (30% vs 50% vs 70% vs 100% Save Rates) — N = {n_all:,}"
)
print("=" * 90)
display(
    cost_summary_df[[
        "Threshold",
        "Targeted",
        "Churners Saved (Recall)",
        "Save Precision (Hit Rate)",
        "Net Saved @ 30% Save",
        "Net Saved @ 50% Save",
        "Net Saved @ 70% Save",
        "Net Saved @ 100% Save",
    ]]
)

# 5. Diagnostic Visualizations
# --- Figure 1: Generalization Metrics across Partitions
fig1, ax1 = plt.subplots(figsize=(9.5, 4.8), dpi=150)
metrics_x = np.arange(len(slice_df))
w = 0.25

ax1.bar(
    metrics_x - w,
    slice_df["ROC-AUC"],
    w,
    label="ROC-AUC",
    color="#2b5c8f",
    edgecolor="black",
    lw=0.7,
)
ax1.bar(
    metrics_x,
    slice_df["PR-AUC"],
    w,
    label="PR-AUC",
    color="#d95f02",
    edgecolor="black",
    lw=0.7,
)
ax1.bar(
    metrics_x + w,
    slice_df["Brier Score"],
    w,
    label="Brier Score",
    color="#10b981",
    edgecolor="black",
    lw=0.7,
)

ax1.set_xticks(metrics_x)
ax1.set_xticklabels(slice_df.index, fontweight="bold", fontsize=9.5)
ax1.set_ylim(0.0, 1.0)
ax1.set_title(
    f"Model Generalization Audit (Out-of-Sample Drop Δ = {auc_gap:+.4f})",
    fontweight="bold",
    fontsize=11.5,
    pad=10,
)
ax1.set_ylabel("Metric Score", fontweight="bold")
ax1.grid(axis="y", ls=":", alpha=0.5)
ax1.legend(loc="upper right", frameon=True, fontsize=8.5)

for i in range(len(slice_df)):
  ax1.text(
      metrics_x[i] - w,
      slice_df["ROC-AUC"].iloc[i] + 0.02,
      f"{slice_df['ROC-AUC'].iloc[i]:.3f}",
      ha="center",
      va="bottom",
      fontsize=8,
      fontweight="bold",
  )
  ax1.text(
      metrics_x[i],
      slice_df["PR-AUC"].iloc[i] + 0.02,
      f"{slice_df['PR-AUC'].iloc[i]:.3f}",
      ha="center",
      va="bottom",
      fontsize=8,
      fontweight="bold",
  )
  ax1.text(
      metrics_x[i] + w,
      slice_df["Brier Score"].iloc[i] + 0.02,
      f"{slice_df['Brier Score'].iloc[i]:.3f}",
      ha="center",
      va="bottom",
      fontsize=8,
      fontweight="bold",
  )

plt.tight_layout()
plt.show()

# --- Figure 2: 4-Panel Grid of Net Capital Preserved across Save Rates
fig2, axes2 = plt.subplots(2, 2, figsize=(16, 10.5), dpi=150)
pol_labels = [p[0].split(". ")[1] for p in policies]
colors_pol = ["#95a5a6", "#e67e22", "#3498db", "#27ae60"]

scenarios_meta = [
    (
        0.30,
        axes2[0, 0],
        "Scenario 1: 30% Save Rate (Severe Resistance / Stress Test)",
    ),
    (0.50, axes2[0, 1], "Scenario 2: 50% Save Rate (Realistic Industry Target)"),
    (0.70, axes2[1, 0], "Scenario 3: 70% Save Rate (High Efficacy Campaign)"),
    (
        1.00,
        axes2[1, 1],
        "Scenario 4: 100% Save Rate (Theoretical Perfect Save)",
    ),
]

for gamma, ax, subtitle in scenarios_meta:
  g_key = int(gamma * 100)
  saved_vals = [r[f"_raw_saved_{g_key}"] for r in policy_rows]

  bars = ax.bar(
      pol_labels,
      saved_vals,
      color=colors_pol,
      edgecolor="black",
      linewidth=0.8,
      width=0.52,
  )
  ax.axhline(0, color="black", lw=1.1, ls="--")
  ax.yaxis.set_major_formatter("${x:,.0f}")
  ax.set_title(subtitle, fontweight="bold", fontsize=11, pad=10)
  ax.set_ylabel("Net Preserved Capital ($)", fontweight="bold")
  ax.grid(axis="y", ls=":", alpha=0.5)
  plt.setp(ax.get_xticklabels(), rotation=12, ha="right", fontweight="bold")

  # Dynamic y-limits with padding for text annotations
  y_min, y_max = min(saved_vals), max(saved_vals)
  pad_top = max(y_max * 0.22, 40000)
  pad_bot = max(abs(y_min) * 0.22, 40000) if y_min < 0 else 20000
  ax.set_ylim(y_min - pad_bot, y_max + pad_top)

  for bar, val in zip(bars, saved_vals):
    if val > 0:
      ax.text(
          bar.get_x() + bar.get_width() / 2,
          val + (y_max * 0.03 if y_max > 0 else 5000),
          f"+${val:,.0f}",
          ha="center",
          va="bottom",
          fontsize=8.5,
          fontweight="bold",
          color="#1b5e20",
      )
    elif val < 0:
      ax.text(
          bar.get_x() + bar.get_width() / 2,
          val - (abs(y_min) * 0.05 if y_min < 0 else 5000),
          f"-${abs(val):,.0f}",
          ha="center",
          va="top",
          fontsize=8.5,
          fontweight="bold",
          color="#b71c1c",
      )
    else:
      ax.text(
          bar.get_x() + bar.get_width() / 2,
          5000,
          "$0",
          ha="center",
          va="bottom",
          fontsize=8.5,
          fontweight="bold",
          color="#7f8c8d",
      )

fig2.suptitle(
    "Net Capital Preserved vs. Null Action across Retention Efficacy Scenarios"
    " (N = 7,043)",
    fontweight="bold",
    fontsize=13.5,
    y=0.995,
)
plt.tight_layout()
plt.show()

# 6. Operational Roster Production & CRM Export
risk_tiers = pd.qcut(
    p_all,
    q=[0.0, 0.50, 0.80, 0.95, 1.0],
    labels=["Low Risk", "Moderate Risk", "High Risk", "Critical Risk"],
)

t_star_vec = (p_all >= t_star).astype(int)

roster_df = pd.DataFrame({
    "CustomerID": df["customerID"].values,
    "Contract": df["Contract"].values,
    "Tenure_Months": df["tenure"].values,
    "MonthlyCharges": df["MonthlyCharges"].values,
    "Actual_Churn": y_all.values,
    "Predicted_Churn_Prob": np.round(p_all, 4),
    "Risk_Tier": risk_tiers,
    f"Target_Action_Policy (t*={t_star:.2f})": t_star_vec,
})

roster_df.to_csv("telco_retention_action_roster.csv", index=False)

print("=" * 86)
print("PRODUCTION DEPLOYMENT & CRM ROSTER GENERATED")
print("=" * 86)
print("Export Path                   : 'telco_retention_action_roster.csv'")
print(f"Total Scored Accounts         : {len(roster_df):,}")
print(
    f"Out-of-Sample AUC Drop (Δ)    : {auc_gap:+.4f} (Under 0.015 threshold -> Verified Minimum Leak)"
)
print(
    f"Cost-Optimal Outreach (t*={t_star:.2f}) : {t_star_vec.sum():,} accounts flagged ({t_star_vec.sum()/n_all*100:.1f}% of cohort)"
)
print(
    f"Churners Rescued              : {cost_summary_df.loc[f'4. Model @ t* = {t_star:.2f}', 'Churners Saved (Recall)']}"
)
print(
    f"Net Value @ 50% Save (Target) : {cost_summary_df.loc[f'4. Model @ t* = {t_star:.2f}', 'Net Saved @ 50% Save']}"
)
print(
    f"Net Value @ 100% Save (Ideal) : {cost_summary_df.loc[f'4. Model @ t* = {t_star:.2f}', 'Net Saved @ 100% Save']}"
)
print("=" * 86)

In [ ]:
import joblib
import json

# 1. Compile Deployment Bundle
export_payload = {
    "model_name": final_model["name"],
    "classifier": final_clf,
    "feature_cols": feature_cols,
    "numeric_cols": NUMERIC_COLS,
    "categorical_cols": CATEGORICAL_COLS,
    "boolean_cols": BOOLEAN_COLS,
    "thresholds": {
        "t_star": float(t_star),       # 0.17 cost-optimal
        "t_default": float(t_def),     # 0.50
    },
    "cost_matrix": {
        "C_FN": C_FN_VAL,
        "C_FP": C_FP_VAL,
    },
    "risk_tiers": {
        "bins": [0.0, 0.50, 0.80, 0.95, 1.0],
        "labels": ["Low Risk", "Moderate Risk", "High Risk", "Critical Risk"]
    }
}

os.makedirs("../artifacts", exist_ok=True)
model_filename = "../artifacts/telco_churn_model_bundle.joblib"
joblib.dump(export_payload, model_filename, compress=3)
print(f"Artifact successfully saved: '{model_filename}'")

# 2. Sanity Test: Load and Verify Inference
loaded_bundle = joblib.load(model_filename)
clf_test = loaded_bundle["classifier"]

# Run a quick check against the first row of X_all
sample_x = X_all.iloc[[0]]
sample_prob = float(clf_test.predict_proba(sample_x)[:, 1][0])
sample_pred = int(sample_prob >= loaded_bundle["thresholds"]["t_star"])

print(f"Verification Check:")
print(f"  • Sample Raw P(Churn) : {sample_prob:.4f}")
print(f"  • Flagged for Action  : {bool(sample_pred)} (t* >= {loaded_bundle['thresholds']['t_star']})")